<a href="https://colab.research.google.com/github/eshan14git/football-qa-nlp/blob/eshan-dev/notebooks/08_LSTM_football_qa_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [113]:
# Cell 1 - Clone the correct GitHub branch

!git clone -b eshan-dev https://github.com/eshan14git/football-qa-nlp.git

fatal: destination path 'football-qa-nlp' already exists and is not an empty directory.


In [114]:
# Cell 2 - Import libraries

import os
import re
import joblib
import pandas as pd

project_path = "/content/football-qa-nlp"
data_path = os.path.join(project_path, "data")
rf_path = os.path.join(project_path, "models", "random_forest")

print("Setup complete.")

Setup complete.


In [115]:
# Chatbot display mode

DEBUG = False

In [116]:

# Move into the repository
%cd /content/football-qa-nlp

# Verify location
import os

print("\nCurrent directory:")
print(os.getcwd())

print("\nLSTM files:")
print(os.listdir("/content/football-qa-nlp/models/lstm"))

/content/football-qa-nlp

Current directory:
/content/football-qa-nlp

LSTM files:
['lstm_label_encoder.pkl', 'lstm_max_sequence_length.pkl', 'lstm_intent_model.keras', 'lstm_tokenizer.pkl']


In [117]:
import os

print("Current directory:")
print(os.getcwd())

print("\nCurrent files/folders:")
print(os.listdir("."))

Current directory:
/content/football-qa-nlp

Current files/folders:
['src', 'README.md', 'football-qa-nlp', '.gitignore', '06_football_qa_inference.ipynb', 'docs', 'data', 'models', 'requirements.txt', '.git', 'notebooks']


In [118]:
# Cell 3 - Load LSTM model and preprocessing artifacts

import os
import pickle
import tensorflow as tf

lstm_path = "models/lstm"

# Load trained LSTM model
lstm_model = tf.keras.models.load_model(
    os.path.join(lstm_path, "lstm_intent_model.keras")
)

# Load tokenizer
with open(
    os.path.join(lstm_path, "lstm_tokenizer.pkl"),
    "rb"
) as file:
    lstm_tokenizer = pickle.load(file)

# Load label encoder
with open(
    os.path.join(lstm_path, "lstm_label_encoder.pkl"),
    "rb"
) as file:
    lstm_label_encoder = pickle.load(file)

# Load max sequence length
with open(
    os.path.join(lstm_path, "lstm_max_sequence_length.pkl"),
    "rb"
) as file:
    LSTM_MAX_SEQUENCE_LENGTH = pickle.load(file)

print("LSTM model loaded.")
print("LSTM tokenizer loaded.")
print("LSTM label encoder loaded.")
print("Max sequence length:", LSTM_MAX_SEQUENCE_LENGTH)

LSTM model loaded.
LSTM tokenizer loaded.
LSTM label encoder loaded.
Max sequence length: 20


In [119]:
# Cell 4 - Load cleaned datasets

results_df = pd.read_csv(
    os.path.join(data_path, "results_clean.csv")
)

goalscorers_df = pd.read_csv(
    os.path.join(data_path, "goalscorers_clean.csv")
)

shootouts_df = pd.read_csv(
    os.path.join(data_path, "shootouts_clean.csv")
)

former_names_df = pd.read_csv(
    os.path.join(data_path, "former_names_clean.csv")
)

print("Datasets loaded successfully.")
print("Results:", results_df.shape)
print("Goalscorers:", goalscorers_df.shape)
print("Shootouts:", shootouts_df.shape)
print("Former names:", former_names_df.shape)

Datasets loaded successfully.
Results: (49485, 9)
Goalscorers: (47855, 8)
Shootouts: (682, 5)
Former names: (36, 4)


In [120]:
# LSTM intent prediction function

import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_intent(question):

    # Convert question to token sequence
    sequence = lstm_tokenizer.texts_to_sequences([str(question)])

    # Apply the same padding used during LSTM training
    padded_sequence = pad_sequences(
        sequence,
        maxlen=LSTM_MAX_SEQUENCE_LENGTH,
        padding="post",
        truncating="post"
    )

    # Predict intent probabilities
    probabilities = lstm_model.predict(
        padded_sequence,
        verbose=0
    )[0]

    # Get class with highest probability
    predicted_index = np.argmax(probabilities)

    # Convert class index back to intent name
    predicted_intent = lstm_label_encoder.inverse_transform(
        [predicted_index]
    )[0]

    return predicted_intent


# Test the LSTM intent predictor
test_question = "Who won the 2014 FIFA World Cup Final?"

print("Question:", test_question)
print("Predicted intent:", predict_intent(test_question))

Question: Who won the 2014 FIFA World Cup Final?
Predicted intent: match_winner


In [121]:
# Cell 6 - Prepare searchable team and tournament lists

all_teams = sorted(
    set(results_df["home_team"].dropna().tolist()) |
    set(results_df["away_team"].dropna().tolist())
)

all_tournaments = sorted(
    results_df["tournament"].dropna().unique().tolist()
)

print("Unique teams:", len(all_teams))
print("Unique tournaments:", len(all_tournaments))

Unique teams: 336
Unique tournaments: 200


In [122]:
# Cell 24 - Text normalization helper

import unicodedata

def normalize_text(text):
    text = str(text).lower().strip()

    # Remove accents:
    # América -> America
    # Côte -> Cote
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        char for char in text
        if not unicodedata.combining(char)
    )

    return text

In [123]:
import unicodedata

def normalize_text(text):
    text = str(text).lower().strip()

    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        char for char in text
        if not unicodedata.combining(char)
    )

    return text

print(normalize_text("Copa América"))
print(normalize_text("Copa America"))

copa america
copa america


In [124]:
# Cell 7 - Extract basic entities from a question
# Improved with whole-name matching and longest-name priority

def extract_basic_entities(question):
    question_normalized = normalize_text(question)

    # -------------------------------------------------
    # 1. Find teams using whole-name matching
    # -------------------------------------------------

    # Longest names first:
    # "DR Congo" should be checked before "Congo"
    # "Nigeria" should be checked before "Niger"
    sorted_teams = sorted(
        all_teams,
        key=lambda x: len(normalize_text(x)),
        reverse=True
    )

    found_teams = []

    for team in sorted_teams:
        team_normalized = normalize_text(team)

        pattern = (
            r"(?<!\w)"
            + re.escape(team_normalized)
            + r"(?!\w)"
        )

        if re.search(pattern, question_normalized):

            # Prevent shorter overlapping team names
            overlap = False

            for already_found in found_teams:
                already_normalized = normalize_text(already_found)

                if (
                    team_normalized in already_normalized
                    and team_normalized != already_normalized
                ):
                    overlap = True
                    break

            if not overlap:
                found_teams.append(team)

    # -------------------------------------------------
    # 2. Find year
    # -------------------------------------------------
    year_match = re.search(
        r"\b(18|19|20)\d{2}\b",
        question
    )

    year = year_match.group() if year_match else None

    # -------------------------------------------------
    # 3. Find tournament
    # -------------------------------------------------
    sorted_tournaments = sorted(
        all_tournaments,
        key=lambda x: len(normalize_text(x)),
        reverse=True
    )

    found_tournaments = []

    for tournament in sorted_tournaments:
        tournament_normalized = normalize_text(tournament)

        pattern = (
            r"(?<!\w)"
            + re.escape(tournament_normalized)
            + r"(?!\w)"
        )

        if re.search(pattern, question_normalized):

            overlap = False

            for already_found in found_tournaments:
                already_normalized = normalize_text(already_found)

                if (
                    tournament_normalized in already_normalized
                    and tournament_normalized != already_normalized
                ):
                    overlap = True
                    break

            if not overlap:
                found_tournaments.append(tournament)

    return {
        "teams": found_teams,
        "year": year,
        "tournaments": found_tournaments
    }

In [125]:
# Cell 8 - Test entity extraction

test_questions = [
    "Who won between India and Myanmar?",
    "Who won between India and Myanmar in 2017?",
    "Who won the FIFA World Cup match between Germany and Argentina in 2014?",
    "What was the score between Brazil and Argentina in Copa América?"
]

for q in test_questions:
    print("\nQuestion:", q)
    print(extract_basic_entities(q))


Question: Who won between India and Myanmar?
{'teams': ['Myanmar', 'India'], 'year': None, 'tournaments': []}

Question: Who won between India and Myanmar in 2017?
{'teams': ['Myanmar', 'India'], 'year': '2017', 'tournaments': []}

Question: Who won the FIFA World Cup match between Germany and Argentina in 2014?
{'teams': ['Argentina', 'Germany'], 'year': '2014', 'tournaments': ['FIFA World Cup']}

Question: What was the score between Brazil and Argentina in Copa América?
{'teams': ['Argentina', 'Brazil'], 'year': None, 'tournaments': ['Copa América']}


In [126]:
# Cell 76 - Detect explicit home/away team constraints

def extract_team_role_constraints(question):
    q = normalize_text(question)

    home_team = None
    away_team = None

    for team in all_teams:
        t = normalize_text(team)

        if f"home team {t}" in q or f"{t} as the home team" in q:
            home_team = team

        if f"away team {t}" in q or f"{t} as the away team" in q:
            away_team = team

    return {
        "home_team": home_team,
        "away_team": away_team
    }

In [127]:
# Cell 78 - Extract exact date from question

def extract_exact_date(question):
    date_match = re.search(
        r"\b(18|19|20)\d{2}-\d{2}-\d{2}\b",
        question
    )

    return date_match.group() if date_match else None

In [128]:
# Cell 9 - Find candidate matches from extracted entities
# Improved with:
# - explicit home/away roles
# - exact-date priority
# - historical/former team-name handling

def find_result_candidates(question):

    # -------------------------------------------------
    # 0. Resolve historical team names
    # -------------------------------------------------
    resolved_question, historical_constraints = (
        resolve_former_team_names(question)
    )

    # Extract entities from the resolved question
    entities = extract_basic_entities(resolved_question)

    candidates = results_df.copy()

    teams = entities["teams"]
    year = entities["year"]
    tournaments = entities["tournaments"]

    # -------------------------------------------------
    # 1. Filter by two detected teams
    # -------------------------------------------------
    if len(teams) >= 2:
        team1, team2 = teams[:2]

        candidates = candidates[
            (
                (candidates["home_team"] == team1) &
                (candidates["away_team"] == team2)
            )
            |
            (
                (candidates["home_team"] == team2) &
                (candidates["away_team"] == team1)
            )
        ]

    # -------------------------------------------------
    # 2. Apply explicit home/away team constraints
    # -------------------------------------------------
    roles = extract_team_role_constraints(resolved_question)

    if roles["home_team"]:
        candidates = candidates[
            candidates["home_team"] == roles["home_team"]
        ]

    if roles["away_team"]:
        candidates = candidates[
            candidates["away_team"] == roles["away_team"]
        ]

    # -------------------------------------------------
    # 3. Exact date first, otherwise year
    # -------------------------------------------------
    exact_date = extract_exact_date(resolved_question)

    if exact_date:
        candidates = candidates[
            candidates["date"].astype(str) == exact_date
        ]

    elif year:
        candidates = candidates[
            candidates["date"]
            .astype(str)
            .str.startswith(year)
        ]

    # -------------------------------------------------
    # 4. Filter by tournament
    # -------------------------------------------------
    if tournaments:
        tournament = tournaments[0]

        candidates = candidates[
            candidates["tournament"]
            .str.lower()
            == tournament.lower()
        ]

    # -------------------------------------------------
    # 5. Apply historical-name validity periods
    # -------------------------------------------------
    if historical_constraints:

        candidate_dates = pd.to_datetime(
            candidates["date"],
            errors="coerce"
        )

        for constraint in historical_constraints:

            start_date = constraint["start_date"]
            end_date = constraint["end_date"]

            candidates = candidates[
                (candidate_dates >= start_date) &
                (candidate_dates <= end_date)
            ]

            # Rebuild dates after filtering
            candidate_dates = pd.to_datetime(
                candidates["date"],
                errors="coerce"
            )

    return candidates

In [129]:
# Cell 90 - Detect historical/former team names

def detect_former_team_names(question):
    q = normalize_text(question)

    matches = []

    # Longest names first to avoid partial-name conflicts
    rows = former_names_lookup.copy()

    rows["former_length"] = (
        rows["former"]
        .astype(str)
        .str.len()
    )

    rows = rows.sort_values(
        "former_length",
        ascending=False
    )

    for _, row in rows.iterrows():
        former = str(row["former"])
        former_normalized = normalize_text(former)

        pattern = (
            r"(?<!\w)"
            + re.escape(former_normalized)
            + r"(?!\w)"
        )

        if re.search(pattern, q):
            matches.append({
                "former": former,
                "current": row["current"],
                "start_date": row["start_date"],
                "end_date": row["end_date"]
            })

    return matches

In [130]:
# Cell 92 - Resolve former team names for dataset searching
# Improved with accent-insensitive replacement

def resolve_former_team_names(question):
    detected = detect_former_team_names(question)

    resolved_question = question
    historical_constraints = []

    for item in detected:
        former = item["former"]
        current = item["current"]

        # -------------------------------------------------
        # Accent-insensitive historical-name replacement
        # -------------------------------------------------
        words = resolved_question.split()

        former_normalized = normalize_text(former)

        # Try progressively sized word sequences
        former_word_count = len(former.split())

        for i in range(len(words) - former_word_count + 1):

            phrase = " ".join(
                words[i:i + former_word_count]
            )

            # Remove surrounding punctuation for comparison
            phrase_clean = re.sub(
                r"^[^\w]+|[^\w]+$",
                "",
                phrase
            )

            if normalize_text(phrase_clean) == former_normalized:

                # Preserve punctuation surrounding the phrase
                pattern = re.escape(phrase)

                resolved_question = re.sub(
                    pattern,
                    current,
                    resolved_question,
                    count=1,
                    flags=re.IGNORECASE
                )

                break

        # Preserve historical validity period
        historical_constraints.append({
            "former": former,
            "current": current,
            "start_date": item["start_date"],
            "end_date": item["end_date"]
        })

    return resolved_question, historical_constraints

In [131]:
# Cell 89 - Prepare historical team-name mappings

former_names_lookup = former_names_df.copy()

former_names_lookup["start_date"] = pd.to_datetime(
    former_names_lookup["start_date"]
)

former_names_lookup["end_date"] = pd.to_datetime(
    former_names_lookup["end_date"]
)

print("Prepared former-name mappings:", len(former_names_lookup))

print("\nDate types:")
print(
    former_names_lookup[
        ["start_date", "end_date"]
    ].dtypes
)

Prepared former-name mappings: 36

Date types:
start_date    datetime64[ns]
end_date      datetime64[ns]
dtype: object


In [132]:
# Cell 10 - Test candidate retrieval

question = "Who won between India and Myanmar in 2017?"

candidates = find_result_candidates(question)

print("Candidate matches found:", len(candidates))

display(
    candidates[
        [
            "date",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
            "tournament",
            "city",
            "country"
        ]
    ]
)

Candidate matches found: 2


,date,home_team,away_team,home_score,away_score,tournament,city,country
40535,2017-03-28,Myanmar,India,0,1,AFC Asian Cup qualification,Yangon,Myanmar
41204,2017-11-14,India,Myanmar,2,2,AFC Asian Cup qualification,Margao,India


In [133]:
# Cell 11 - Inspect differences between candidate matches

def inspect_candidate_differences(candidates):
    if len(candidates) <= 1:
        return {}

    differences = {}

    fields = [
        "date",
        "tournament",
        "city",
        "country",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "neutral"
    ]

    for field in fields:
        unique_values = candidates[field].dropna().astype(str).unique().tolist()

        if len(unique_values) > 1:
            differences[field] = unique_values

    return differences

In [134]:
# Cell 12 - Test ambiguity inspection

question = "Who won between India and Myanmar in 2017?"

candidates = find_result_candidates(question)

differences = inspect_candidate_differences(candidates)

print("Candidate matches:", len(candidates))
print("\nDifferences:")

for field, values in differences.items():
    print(f"- {field}: {values}")

Candidate matches: 2

Differences:
- date: ['2017-03-28', '2017-11-14']
- city: ['Yangon', 'Margao']
- country: ['Myanmar', 'India']
- home_team: ['Myanmar', 'India']
- away_team: ['India', 'Myanmar']
- home_score: ['0', '2']
- away_score: ['1', '2']


In [135]:
# Cell 13 - Generate a clarification question

def generate_clarification_question(candidates):
    if len(candidates) == 0:
        return "I couldn't find a matching game."

    if len(candidates) == 1:
        return None

    differences = inspect_candidate_differences(candidates)

    # Prefer tournament when tournaments differ
    if "tournament" in differences:
        options = differences["tournament"]
        return (
            "I found multiple matching games. "
            "Do you remember the tournament? "
            + "Options: "
            + ", ".join(options)
        )

    # Then country/location
    if "country" in differences:
        options = differences["country"]
        return (
            "I found multiple matching games. "
            "Do you remember which country the game was played in? "
            + "Options: "
            + ", ".join(options)
        )

    # Then city
    if "city" in differences:
        options = differences["city"]
        return (
            "I found multiple matching games. "
            "Do you remember the city? "
            + "Options: "
            + ", ".join(options)
        )

    # Then home team
    if "home_team" in differences:
        options = differences["home_team"]
        return (
            "I found multiple matching games. "
            "Do you remember which team was the home team? "
            + "Options: "
            + ", ".join(options)
        )

    # Then exact date only as a last resort
    if "date" in differences:
        options = differences["date"]
        return (
            "I found multiple matching games. "
            "Do you remember which date it was? "
            + "Options: "
            + ", ".join(options)
        )

    return (
        "I found multiple matching games, but I need one more detail "
        "to identify the correct one."
    )

In [136]:
# Cell 14 - Test clarification generation

question = "Who won between India and Myanmar in 2017?"

candidates = find_result_candidates(question)

clarification = generate_clarification_question(candidates)

print("Question:", question)
print("\nAssistant:", clarification)

Question: Who won between India and Myanmar in 2017?

Assistant: I found multiple matching games. Do you remember which country the game was played in? Options: Myanmar, India


In [137]:
question = "Who won between India and Myanmar in 2017?"

candidates = find_result_candidates(question)

print("Candidate matches:", len(candidates))
print(generate_clarification_question(candidates))

Candidate matches: 2
I found multiple matching games. Do you remember which country the game was played in? Options: Myanmar, India


In [138]:
# Cell 15 - Apply a clarification reply to existing candidates

def apply_clarification(candidates, reply):
    reply_lower = reply.lower().strip()

    filtered = candidates.copy()

    # Year
    year_match = re.search(r"\b(18|19|20)\d{2}\b", reply)
    if year_match:
        year = year_match.group()
        filtered = filtered[
            filtered["date"].astype(str).str.startswith(year)
        ]

    # Tournament
    tournament_matches = [
        tournament
        for tournament in filtered["tournament"].dropna().unique()
        if tournament.lower() in reply_lower
    ]

    if tournament_matches:
        tournament = tournament_matches[0]
        filtered = filtered[
            filtered["tournament"].str.lower() == tournament.lower()
        ]

    # Country
    country_matches = [
        country
        for country in filtered["country"].dropna().unique()
        if str(country).lower() in reply_lower
    ]

    if country_matches:
        country = country_matches[0]
        filtered = filtered[
            filtered["country"].astype(str).str.lower() == str(country).lower()
        ]

    # City
    city_matches = [
        city
        for city in filtered["city"].dropna().unique()
        if str(city).lower() in reply_lower
    ]

    if city_matches:
        city = city_matches[0]
        filtered = filtered[
            filtered["city"].astype(str).str.lower() == str(city).lower()
        ]

    # Home team
    home_team_matches = [
        team
        for team in filtered["home_team"].dropna().unique()
        if str(team).lower() in reply_lower
    ]

    if home_team_matches:
        team = home_team_matches[0]
        filtered = filtered[
            filtered["home_team"].astype(str).str.lower() == str(team).lower()
        ]

    return filtered

In [139]:
# Cell 16 - Test clarification filtering

question = "Who won between India and Myanmar in 2017?"

candidates = find_result_candidates(question)

print("Initial candidates:", len(candidates))

clarification_reply = "India"

filtered_candidates = apply_clarification(
    candidates,
    clarification_reply
)

print("Candidates after clarification:", len(filtered_candidates))

display(
    filtered_candidates[
        [
            "date",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
            "tournament",
            "city",
            "country"
        ]
    ]
)

Initial candidates: 2
Candidates after clarification: 1


,date,home_team,away_team,home_score,away_score,tournament,city,country
41204,2017-11-14,India,Myanmar,2,2,AFC Asian Cup qualification,Margao,India


In [140]:
# Cell 17 - Generate an answer from one selected result row

def generate_result_answer(intent, row):
    home_team = row["home_team"]
    away_team = row["away_team"]
    home_score = row["home_score"]
    away_score = row["away_score"]
    date = row["date"]
    tournament = row["tournament"]
    city = row["city"]
    country = row["country"]
    neutral = row["neutral"]

    if intent == "home_team_score":
        return f"{home_team} scored {home_score} goal(s)."

    elif intent == "away_team_score":
        return f"{away_team} scored {away_score} goal(s)."

    elif intent == "match_score":
        return f"{home_team} {home_score} - {away_score} {away_team}."

    elif intent == "match_winner":
        if home_score > away_score:
            return f"{home_team} won the match {home_score}-{away_score}."
        elif away_score > home_score:
            return f"{away_team} won the match {away_score}-{home_score}."
        else:
            return f"The match ended in a {home_score}-{away_score} draw."

    elif intent == "total_goals":
        total = home_score + away_score
        return f"A total of {total} goal(s) were scored."

    elif intent == "match_date":
        return f"The match was played on {date}."

    elif intent == "match_location":
        return f"The match was played in {city}, {country}."

    elif intent == "tournament":
        return f"The match was part of the {tournament}."

    elif intent == "neutral_status":
        neutral_text = "Yes" if bool(neutral) else "No"
        return f"{neutral_text}, the match {'was' if bool(neutral) else 'was not'} played at a neutral venue."

    return "Answer generation for this intent is not implemented yet."

In [141]:
# Cell 18 - Test result answer generation

question = "Who won between India and Myanmar in 2017?"

intent = predict_intent(question)

candidates = find_result_candidates(question)

print("Intent:", intent)
print("Initial candidates:", len(candidates))

# User clarification
reply = "India"

filtered_candidates = apply_clarification(
    candidates,
    reply
)

print("Candidates after clarification:", len(filtered_candidates))

if len(filtered_candidates) == 1:
    selected_match = filtered_candidates.iloc[0]
    answer = generate_result_answer(intent, selected_match)
    print("Answer:", answer)
else:
    print(generate_clarification_question(filtered_candidates))

Intent: match_winner
Initial candidates: 2
Candidates after clarification: 1
Answer: The match ended in a 2-2 draw.


# Cell 19 - Interactive Results QA conversation

RESULT_INTENTS = {
    "home_team_score",
    "away_team_score",
    "match_date",
    "match_location",
    "match_score",
    "match_winner",
    "neutral_status",
    "total_goals",
    "tournament"
}


def ask_results_question(question):
    # Step 1: Predict intent
    intent = predict_intent(question)

    #print(f"\nPredicted intent: {intent}")

    # Make sure this function is only handling results-based intents
    if intent not in RESULT_INTENTS:
        print(
            "This question belongs to another data source "
            "and will be handled in a later stage."
        )
        return

    # Step 2: Find initial candidate matches
    candidates = find_result_candidates(question)

    print(f"Matching records found: {len(candidates)}")

    # Nothing found
    if len(candidates) == 0:
        print(
            "\nAssistant: I couldn't find a match that fits "
            "the information in your question."
        )
        return

    # Step 3: Resolve ambiguity conversationally
    while len(candidates) > 1:

        clarification = generate_smart_clarification(candidates)

        print(f"\nAssistant: {clarification}")

        reply = input("\nYou: ").strip()

        if not reply:
            print("\nAssistant: Please provide a little more information.")
            continue

        new_candidates = apply_clarification(candidates, reply)

        # Clarification didn't narrow anything
        if len(new_candidates) == len(candidates):
            print(
                "\nAssistant: That didn't narrow the matches down. "
                "Let's try another detail."
            )
            continue

        # Clarification accidentally removed everything
        if len(new_candidates) == 0:
            print(
                "\nAssistant: I couldn't match that detail to the "
                "remaining games. Please try another detail."
            )
            continue

        candidates = new_candidates

        print(f"\nRemaining matches: {len(candidates)}")

    # Step 4: Exactly one match remains
    selected_match = candidates.iloc[0]

    answer = generate_result_answer(
        intent,
        selected_match
    )

    print(f"\nAssistant: {answer}")

In [142]:
# Cell 21 - Smarter clarification generator

def generate_smart_clarification(candidates):
    if len(candidates) == 0:
        return "I couldn't find a matching game."

    if len(candidates) == 1:
        return None

    # Build useful derived fields
    temp = candidates.copy()
    temp["year"] = temp["date"].astype(str).str[:4]

    # 1. Prefer year if multiple years remain
    years = sorted(temp["year"].dropna().unique().tolist())

    if len(years) > 1:
        # Avoid dumping too many options
        if len(years) <= 8:
            return (
                "I found matches from multiple years. "
                "Do you remember the year? "
                f"Options: {', '.join(years)}"
            )

        return (
            f"I found matches across {len(years)} different years. "
            "Do you remember roughly which year it was?"
        )

    # 2. Tournament
    tournaments = sorted(
        temp["tournament"].dropna().astype(str).unique().tolist()
    )

    if len(tournaments) > 1:
        if len(tournaments) <= 8:
            return (
                "I found more than one possible match. "
                "Do you remember the tournament? "
                f"Options: {', '.join(tournaments)}"
            )

        return (
            "I found matches from several tournaments. "
            "Do you remember which competition it was?"
        )

    # 3. Country
    countries = sorted(
        temp["country"].dropna().astype(str).unique().tolist()
    )

    if len(countries) > 1:
        return (
            "Do you remember which country the match was played in? "
            f"Options: {', '.join(countries)}"
        )

    # 4. City
    cities = sorted(
        temp["city"].dropna().astype(str).unique().tolist()
    )

    if len(cities) > 1:
        return (
            "Do you remember the city where the match was played? "
            f"Options: {', '.join(cities)}"
        )

    # 5. Home team
    home_teams = sorted(
        temp["home_team"].dropna().astype(str).unique().tolist()
    )

    if len(home_teams) > 1:
        return (
            "Do you remember which team was listed as the home team? "
            f"Options: {', '.join(home_teams)}"
        )

    # 6. Exact date only as a last resort
    dates = sorted(
        temp["date"].dropna().astype(str).unique().tolist()
    )

    if len(dates) > 1:
        return (
            "I still found more than one possible match. "
            "Do you remember the exact date? "
            f"Options: {', '.join(dates)}"
        )

    return (
        "I still found multiple matching records. "
        "Can you give me one more detail about the match?"
    )

In [143]:
print("Model loaded:", "rf_model" in globals())
print("Results loaded:", "results_df" in globals())
print("Chatbot loaded:", "ask_results_question" in globals())
print("Smart clarification loaded:", "generate_smart_clarification" in globals())

Model loaded: False
Results loaded: True
Chatbot loaded: True
Smart clarification loaded: True


In [144]:
# Improved natural clarification filtering
# with accent normalization and exact-date priority

def apply_smart_clarification(candidates, reply):
    reply_normalized = normalize_text(reply)
    filtered = candidates.copy()

    # -------------------------------------------------
    # 0. EXACT DATE - MUST COME BEFORE YEAR
    # -------------------------------------------------
    exact_date = extract_exact_date(reply)

    if exact_date:
        candidate_dates = pd.to_datetime(
            filtered["date"],
            errors="coerce"
        ).dt.strftime("%Y-%m-%d")

        date_filtered = filtered[
            candidate_dates == exact_date
        ]

        if not date_filtered.empty:
            return date_filtered

    # -------------------------------------------------
    # 1. YEAR
    # -------------------------------------------------
    year_match = re.search(
        r"\b(18|19|20)\d{2}\b",
        reply
    )

    if year_match:
        year = year_match.group()

        candidate_years = pd.to_datetime(
            filtered["date"],
            errors="coerce"
        ).dt.strftime("%Y")

        year_filtered = filtered[
            candidate_years == year
        ]

        if not year_filtered.empty:
            return year_filtered

    # -------------------------------------------------
    # 2. EXACT / NATURAL TOURNAMENT MATCH
    # -------------------------------------------------
    tournaments = (
        filtered["tournament"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    for tournament in tournaments:
        tournament_normalized = normalize_text(tournament)

        if tournament_normalized in reply_normalized:
            tournament_filtered = filtered[
                filtered["tournament"]
                .apply(normalize_text)
                == tournament_normalized
            ]

            if not tournament_filtered.empty:
                return tournament_filtered

    # -------------------------------------------------
    # 3. PARTIAL TOURNAMENT MATCH
    # -------------------------------------------------
    meaningful_words = [
        word
        for word in re.findall(
            r"[a-zA-Z]+",
            reply_normalized
        )
        if len(word) >= 4
    ]

    if meaningful_words:
        tournament_mask = (
            filtered["tournament"]
            .fillna("")
            .apply(
                lambda value: all(
                    word in normalize_text(value)
                    for word in meaningful_words
                )
            )
        )

        tournament_filtered = filtered[
            tournament_mask
        ]

        if not tournament_filtered.empty:
            return tournament_filtered

    # -------------------------------------------------
    # 4. COUNTRY
    # -------------------------------------------------
    countries = (
        filtered["country"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    for country in countries:
        country_normalized = normalize_text(country)

        if country_normalized in reply_normalized:
            country_filtered = filtered[
                filtered["country"]
                .apply(normalize_text)
                == country_normalized
            ]

            if not country_filtered.empty:
                return country_filtered

    # -------------------------------------------------
    # 5. CITY
    # -------------------------------------------------
    cities = (
        filtered["city"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    for city in cities:
        city_normalized = normalize_text(city)

        if city_normalized in reply_normalized:
            city_filtered = filtered[
                filtered["city"]
                .apply(normalize_text)
                == city_normalized
            ]

            if not city_filtered.empty:
                return city_filtered

    # -------------------------------------------------
    # 6. HOME TEAM
    # -------------------------------------------------
    if "home" in reply_normalized:
        home_teams = (
            filtered["home_team"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        for team in home_teams:
            team_normalized = normalize_text(team)

            if team_normalized in reply_normalized:
                home_filtered = filtered[
                    filtered["home_team"]
                    .apply(normalize_text)
                    == team_normalized
                ]

                if not home_filtered.empty:
                    return home_filtered

    # -------------------------------------------------
    # Nothing understood
    # -------------------------------------------------
    return filtered

In [145]:
question = "Who won between Brazil and Argentina?"

candidates = find_result_candidates(question)

print("Initial candidates:", len(candidates))

test_reply = "around 2021"

filtered = apply_smart_clarification(
    candidates,
    test_reply
)

print("Candidates after reply:", len(filtered))

display(
    filtered[
        [
            "date",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
            "tournament",
            "city",
            "country"
        ]
    ]
)

Initial candidates: 110
Candidates after reply: 2


,date,home_team,away_team,home_score,away_score,tournament,city,country
44234,2021-07-10,Brazil,Argentina,0,1,Copa América,Rio de Janeiro,Brazil
44768,2021-11-16,Argentina,Brazil,0,0,FIFA World Cup qualification,San Juan,Argentina


In [146]:
reply = "the Copa America one"

filtered_again = apply_smart_clarification(
    filtered,
    reply
)

print("Candidates after tournament clarification:", len(filtered_again))

display(
    filtered_again[
        [
            "date",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
            "tournament",
            "city",
            "country"
        ]
    ]
)

Candidates after tournament clarification: 1


,date,home_team,away_team,home_score,away_score,tournament,city,country
44234,2021-07-10,Brazil,Argentina,0,1,Copa América,Rio de Janeiro,Brazil


In [147]:
# Cell 26 - Interactive Results chatbot with smart clarification
# Final version with invalid-year handling and optional debug output

RESULT_INTENTS = {
    "home_team_score",
    "away_team_score",
    "match_date",
    "match_location",
    "match_score",
    "match_winner",
    "neutral_status",
    "total_goals",
    "tournament"
}


def ask_results_question(question):
    # Predict intent
    intent = predict_intent(question)

    if DEBUG:
        print(f"\nPredicted intent: {intent}")

    if intent not in RESULT_INTENTS:
        print(
            "\nAssistant: This question belongs to another "
            "data source and will be handled later."
        )
        return

    # Find initial candidates
    candidates = find_result_candidates(question)

    if DEBUG:
        print(f"Matching records found: {len(candidates)}")

    # No matching records
    if len(candidates) == 0:
        print(
            "\nAssistant: I couldn't find a match that fits "
            "the information in your question."
        )
        return

    # -------------------------------------------------
    # Resolve ambiguity conversationally
    # -------------------------------------------------
    while len(candidates) > 1:

        clarification = generate_smart_clarification(candidates)

        print(f"\nAssistant: {clarification}")

        reply = input("\nYou: ").strip()

        if not reply:
            print(
                "\nAssistant: Please provide a little more information."
            )
            continue

        # -------------------------------------------------
        # Check for an invalid year clarification
        # -------------------------------------------------
        reply_year = extract_year_from_reply(reply)

        if reply_year:
            available_years = sorted(
                candidates["date"]
                .astype(str)
                .str[:4]
                .unique()
                .tolist()
            )

            if reply_year not in available_years:
                print(
                    f"\nAssistant: I couldn't find a matching record "
                    f"from {reply_year}. Please try one of the "
                    f"remaining years: {', '.join(available_years)}."
                )
                continue

        # -------------------------------------------------
        # Apply smart clarification
        # -------------------------------------------------
        new_candidates = apply_smart_clarification(
            candidates,
            reply
        )

        # Reply did not narrow candidates
        if len(new_candidates) == len(candidates):
            print(
                "\nAssistant: That detail didn't narrow the matches down. "
                "Please try another detail."
            )
            continue

        # Reply removed every candidate
        if len(new_candidates) == 0:
            print(
                "\nAssistant: I couldn't match that detail to the "
                "remaining games. Please try another detail."
            )
            continue

        candidates = new_candidates

        if DEBUG:
            print(f"\nRemaining matches: {len(candidates)}")

    # -------------------------------------------------
    # One exact match remains
    # -------------------------------------------------
    selected_match = candidates.iloc[0]

    answer = generate_result_answer(
        intent,
        selected_match
    )

    print(f"\nAssistant: {answer}")

In [148]:
# Cell 79 - Detect year in clarification reply

def extract_year_from_reply(reply):
    year_match = re.search(
        r"\b(18|19|20)\d{2}\b",
        str(reply)
    )

    return year_match.group() if year_match else None

In [149]:

question = "Who won between Brazil and Argentina?"
ask_results_question(question)


Assistant: I found matches across 64 different years. Do you remember roughly which year it was?

You: 2025

Assistant: Argentina won the match 4-1.


In [150]:
# Cell 27 - Full smart conversation test

question = "Who won between Brazil and Argentina?"

ask_results_question(question)


Assistant: I found matches across 64 different years. Do you remember roughly which year it was?

You: 2021

Assistant: I found more than one possible match. Do you remember the tournament? Options: Copa América, FIFA World Cup qualification

You: copa america

Assistant: Argentina won the match 1-0.


In [151]:
GOALSCORER_INTENTS = {
    "scorer",
    "goal_minute",
    "own_goal_status",
    "penalty_status"
}

In [152]:
all_scorers = sorted(
    goalscorers_df["scorer"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

def add_match_key(df):
    temp = df.copy()

    temp["match_key"] = (
        temp["date"].astype(str)
        + "||"
        + temp["home_team"].astype(str)
        + "||"
        + temp["away_team"].astype(str)
    )

    return temp

goalscorers_with_key = add_match_key(goalscorers_df)
results_with_key = add_match_key(results_df)

print("Unique scorers:", len(all_scorers))

Unique scorers: 15360


In [153]:
def extract_goalscorer_entities(question):
    question_normalized = normalize_text(question)

    found_teams = [
        team
        for team in all_teams
        if normalize_text(team) in question_normalized
    ]

    found_players = [
        player
        for player in all_scorers
        if normalize_text(player) in question_normalized
    ]

    year_match = re.search(r"\b(18|19|20)\d{2}\b", question)
    year = year_match.group() if year_match else None

    return {
        "teams": found_teams,
        "players": found_players,
        "year": year
    }

In [154]:
# Cell 35 - Detect the scoring team from phrases like "for Paraguay"

def extract_scoring_team(question):
    question_normalized = normalize_text(question)

    # Look for patterns like:
    # "scored for Paraguay"
    # "goalscorer for Argentina"
    # "goal for Brazil"
    for team in all_teams:
        team_normalized = normalize_text(team)

        patterns = [
            f"for {team_normalized}",
            f"by {team_normalized}"
        ]

        if any(pattern in question_normalized for pattern in patterns):
            return team

    return None

In [155]:
# Cell 36 - Improved goalscorer candidate retrieval

def find_goalscorer_candidates(question):
    entities = extract_goalscorer_entities(question)

    candidates = goalscorers_with_key.copy()

    teams = entities["teams"]
    players = entities["players"]
    year = entities["year"]

    # -------------------------------------------------
    # 1. PLAYER FILTER
    # -------------------------------------------------
    if players:
        player = players[0]

        candidates = candidates[
            candidates["scorer"] == player
        ]

    # -------------------------------------------------
    # 2. TEAM / OPPONENT FILTERING
    # -------------------------------------------------

    # Two teams explicitly mentioned
    if len(teams) >= 2:
        team1, team2 = teams[:2]

        candidates = candidates[
            (
                (candidates["home_team"] == team1)
                & (candidates["away_team"] == team2)
            )
            |
            (
                (candidates["home_team"] == team2)
                & (candidates["away_team"] == team1)
            )
        ]

    # Only one team mentioned + player mentioned
    # Treat the team as the opponent/match participant
    elif len(teams) == 1 and players:
        mentioned_team = teams[0]

        candidates = candidates[
            (candidates["home_team"] == mentioned_team)
            |
            (candidates["away_team"] == mentioned_team)
        ]

    # -------------------------------------------------
    # 3. YEAR
    # -------------------------------------------------
    if year:
        candidates = candidates[
            candidates["date"].astype(str).str.startswith(year)
        ]

    # -------------------------------------------------
    # 4. SCORING TEAM
    # For wording like "scored for Paraguay"
    # -------------------------------------------------
    scoring_team = extract_scoring_team(question)

    if scoring_team:
        candidates = candidates[
            candidates["team"] == scoring_team
        ]

    return candidates

In [156]:
question = "Which player scored for Paraguay against Chile?"

intent = predict_intent(question)
candidates = find_goalscorer_candidates(question)

print("Predicted intent:", intent)
print("Goal records found:", len(candidates))
print("Scoring teams found:", candidates["team"].unique())

display(
    candidates[
        [
            "date",
            "home_team",
            "away_team",
            "team",
            "scorer",
            "minute",
            "own_goal",
            "penalty"
        ]
    ].head(20)
)

Predicted intent: scorer
Goal records found: 68
Scoring teams found: ['Paraguay']


,date,home_team,away_team,team,scorer,minute,own_goal,penalty
106,1922-10-05,Chile,Paraguay,Paraguay,Julio Ramírez,5,False,False
107,1922-10-05,Chile,Paraguay,Paraguay,Ildefonso López,78,False,False
108,1922-10-05,Chile,Paraguay,Paraguay,Luis Fretes,86,False,False
244,1924-11-01,Chile,Paraguay,Paraguay,Ildefonso López,15,False,False
245,1924-11-01,Chile,Paraguay,Paraguay,Ildefonso López,33,False,False
246,1924-11-01,Chile,Paraguay,Paraguay,Gerardo Rivas,52,False,False
323,1926-11-03,Chile,Paraguay,Paraguay,Luis Vargas Peña,40,False,False
941,1937-01-17,Chile,Paraguay,Paraguay,Juan Amarilla,5,False,False
944,1937-01-17,Chile,Paraguay,Paraguay,Martín Flor,47,False,False
945,1937-01-17,Chile,Paraguay,Paraguay,Raúl Núñez Velloso,78,False,False


In [157]:
# Cell 33 - Inspect unique matches represented by goal records

unique_matches = (
    candidates[
        [
            "match_key",
            "date",
            "home_team",
            "away_team"
        ]
    ]
    .drop_duplicates()
    .sort_values("date")
)

print("Goal-event records:", len(candidates))
print("Unique matches:", len(unique_matches))

display(unique_matches.head(20))

Goal-event records: 68
Unique matches: 32


,match_key,date,home_team,away_team
106,1922-10-05||Chile||Paraguay,1922-10-05,Chile,Paraguay
244,1924-11-01||Chile||Paraguay,1924-11-01,Chile,Paraguay
323,1926-11-03||Chile||Paraguay,1926-11-03,Chile,Paraguay
941,1937-01-17||Chile||Paraguay,1937-01-17,Chile,Paraguay
1161,1939-01-15||Chile||Paraguay,1939-01-15,Chile,Paraguay
1287,1942-01-22||Chile||Paraguay,1942-01-22,Chile,Paraguay
1423,1946-01-19||Chile||Paraguay,1946-01-19,Chile,Paraguay
1556,1947-12-23||Chile||Paraguay,1947-12-23,Chile,Paraguay
1672,1949-04-27||Chile||Paraguay,1949-04-27,Chile,Paraguay
1935,1953-02-25||Chile||Paraguay,1953-02-25,Chile,Paraguay


In [158]:
# Cell 34 - Enrich candidate matches using results dataset

enriched_matches = unique_matches.merge(
    results_with_key[
        [
            "match_key",
            "home_score",
            "away_score",
            "tournament",
            "city",
            "country",
            "neutral"
        ]
    ],
    on="match_key",
    how="left"
)

print("Enriched matches:", len(enriched_matches))

display(
    enriched_matches[
        [
            "date",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
            "tournament",
            "city",
            "country"
        ]
    ].head(20)
)

Enriched matches: 32


,date,home_team,away_team,home_score,away_score,tournament,city,country
0,1922-10-05,Chile,Paraguay,0,3,Copa América,Rio de Janeiro,Brazil
1,1924-11-01,Chile,Paraguay,1,3,Copa América,Montevideo,Uruguay
2,1926-11-03,Chile,Paraguay,5,1,Copa América,Santiago,Chile
3,1937-01-17,Chile,Paraguay,2,3,Copa América,Buenos Aires,Argentina
4,1939-01-15,Chile,Paraguay,1,5,Copa América,Lima,Peru
5,1942-01-22,Chile,Paraguay,0,2,Copa América,Montevideo,Uruguay
6,1946-01-19,Chile,Paraguay,2,1,Copa América,Buenos Aires,Argentina
7,1947-12-23,Chile,Paraguay,0,1,Copa América,Guayaquil,Ecuador
8,1949-04-27,Chile,Paraguay,2,4,Copa América,São Paulo,Brazil
9,1953-02-25,Chile,Paraguay,0,3,Copa América,Lima,Peru


In [159]:
test_questions = [
    "Which player scored for Paraguay against Chile?",
    "Who scored for Argentina against France?",
    "Who scored in Paraguay vs Chile?"
]

for q in test_questions:
    print(q)
    print("Scoring team:", extract_scoring_team(q))
    print()

Which player scored for Paraguay against Chile?
Scoring team: Paraguay

Who scored for Argentina against France?
Scoring team: Argentina

Who scored in Paraguay vs Chile?
Scoring team: None



In [160]:
# Cell 38 - Group goal events by unique match

def get_unique_goal_matches(candidates):
    return (
        candidates[
            [
                "match_key",
                "date",
                "home_team",
                "away_team"
            ]
        ]
        .drop_duplicates()
        .sort_values("date")
    )


question = "Which player scored for Paraguay against Chile?"

candidates = find_goalscorer_candidates(question)
unique_goal_matches = get_unique_goal_matches(candidates)

print("Goal records:", len(candidates))
print("Unique matches:", len(unique_goal_matches))

display(unique_goal_matches.head(20))

Goal records: 68
Unique matches: 32


,match_key,date,home_team,away_team
106,1922-10-05||Chile||Paraguay,1922-10-05,Chile,Paraguay
244,1924-11-01||Chile||Paraguay,1924-11-01,Chile,Paraguay
323,1926-11-03||Chile||Paraguay,1926-11-03,Chile,Paraguay
941,1937-01-17||Chile||Paraguay,1937-01-17,Chile,Paraguay
1161,1939-01-15||Chile||Paraguay,1939-01-15,Chile,Paraguay
1287,1942-01-22||Chile||Paraguay,1942-01-22,Chile,Paraguay
1423,1946-01-19||Chile||Paraguay,1946-01-19,Chile,Paraguay
1556,1947-12-23||Chile||Paraguay,1947-12-23,Chile,Paraguay
1672,1949-04-27||Chile||Paraguay,1949-04-27,Chile,Paraguay
1935,1953-02-25||Chile||Paraguay,1953-02-25,Chile,Paraguay


In [161]:
# Cell 39 - Enrich scorer match candidates with results data

def enrich_goal_matches(unique_matches):
    return unique_matches.merge(
        results_with_key[
            [
                "match_key",
                "home_score",
                "away_score",
                "tournament",
                "city",
                "country",
                "neutral"
            ]
        ],
        on="match_key",
        how="left"
    )


enriched_goal_matches = enrich_goal_matches(unique_goal_matches)

print("Enriched unique matches:", len(enriched_goal_matches))

display(
    enriched_goal_matches[
        [
            "date",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
            "tournament",
            "city",
            "country"
        ]
    ].head(20)
)

Enriched unique matches: 32


,date,home_team,away_team,home_score,away_score,tournament,city,country
0,1922-10-05,Chile,Paraguay,0,3,Copa América,Rio de Janeiro,Brazil
1,1924-11-01,Chile,Paraguay,1,3,Copa América,Montevideo,Uruguay
2,1926-11-03,Chile,Paraguay,5,1,Copa América,Santiago,Chile
3,1937-01-17,Chile,Paraguay,2,3,Copa América,Buenos Aires,Argentina
4,1939-01-15,Chile,Paraguay,1,5,Copa América,Lima,Peru
5,1942-01-22,Chile,Paraguay,0,2,Copa América,Montevideo,Uruguay
6,1946-01-19,Chile,Paraguay,2,1,Copa América,Buenos Aires,Argentina
7,1947-12-23,Chile,Paraguay,0,1,Copa América,Guayaquil,Ecuador
8,1949-04-27,Chile,Paraguay,2,4,Copa América,São Paulo,Brazil
9,1953-02-25,Chile,Paraguay,0,3,Copa América,Lima,Peru


In [162]:
# Cell 40 - Clarification for scorer match candidates

def generate_goal_match_clarification(enriched_matches):
    if len(enriched_matches) == 0:
        return "I couldn't find a matching game."

    if len(enriched_matches) == 1:
        return None

    return generate_smart_clarification(enriched_matches)

In [163]:
# Cell 41 - Apply clarification to enriched scorer matches

def apply_goal_match_clarification(enriched_matches, reply):
    return apply_smart_clarification(
        enriched_matches,
        reply
    )

In [164]:
question = "Which player scored for Paraguay against Chile?"

goal_candidates = find_goalscorer_candidates(question)

unique_goal_matches = get_unique_goal_matches(
    goal_candidates
)

enriched_goal_matches = enrich_goal_matches(
    unique_goal_matches
)

print("Possible matches:", len(enriched_goal_matches))

clarification = generate_goal_match_clarification(
    enriched_goal_matches
)

print("\nAssistant:", clarification)

Possible matches: 32

Assistant: I found matches across 29 different years. Do you remember roughly which year it was?


In [165]:
reply = "around 2021"

filtered_goal_matches = apply_goal_match_clarification(
    enriched_goal_matches,
    reply
)

print("Remaining matches:", len(filtered_goal_matches))

display(
    filtered_goal_matches[
        [
            "date",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
            "tournament",
            "city",
            "country"
        ]
    ]
)

Remaining matches: 1


,date,home_team,away_team,home_score,away_score,tournament,city,country
30,2021-06-24,Chile,Paraguay,0,2,Copa América,Brasília,Brazil


In [166]:
# Cell 44 - Retrieve goal events for the selected match

selected_match = filtered_goal_matches.iloc[0]
selected_match_key = selected_match["match_key"]

selected_goal_events = goal_candidates[
    goal_candidates["match_key"] == selected_match_key
].copy()

print("Goal records for selected match:", len(selected_goal_events))

display(
    selected_goal_events[
        [
            "date",
            "home_team",
            "away_team",
            "team",
            "scorer",
            "minute",
            "own_goal",
            "penalty"
        ]
    ]
)

Goal records for selected match: 2


,date,home_team,away_team,team,scorer,minute,own_goal,penalty
40924,2021-06-24,Chile,Paraguay,Paraguay,Braian Samudio,33,False,False
40925,2021-06-24,Chile,Paraguay,Paraguay,Miguel Almirón,58,False,True


In [167]:
# Convert goal minutes to natural ordinal form
# 1 -> 1st, 2 -> 2nd, 3 -> 3rd, 21 -> 21st, etc.

def format_ordinal_minute(minute):
    minute = str(minute).strip()

    # Handle stoppage time such as 90+1
    if "+" in minute:
        return minute

    try:
        number = int(float(minute))
    except ValueError:
        return minute

    if 10 <= number % 100 <= 20:
        suffix = "th"
    else:
        suffix = {
            1: "st",
            2: "nd",
            3: "rd"
        }.get(number % 10, "th")

    return f"{number}{suffix}"

In [168]:
# Cell 45 - Generate answers for goalscorer-based intents
# Improved multiple-goal handling + natural ordinal minutes

def generate_goalscorer_answer(intent, goal_events):
    if goal_events.empty:
        return "I couldn't find a matching goal record."

    home_team = goal_events.iloc[0]["home_team"]
    away_team = goal_events.iloc[0]["away_team"]
    date = goal_events.iloc[0]["date"]

    # -------------------------------------------------
    # SCORER
    # -------------------------------------------------
    if intent == "scorer":
        scorers = (
            goal_events["scorer"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        if len(scorers) == 1:
            return (
                f"{scorers[0]} scored in the "
                f"{home_team} vs {away_team} match on {date}."
            )

        return (
            f"The scorers were {', '.join(scorers)} "
            f"in the {home_team} vs {away_team} match on {date}."
        )

    # -------------------------------------------------
    # GOAL MINUTE
    # Improved for braces / hat-tricks / multiple goals
    # + correct ordinal formatting
    # -------------------------------------------------
    elif intent == "goal_minute":

        scorer_minutes = {}

        for _, row in goal_events.iterrows():
            scorer = str(row["scorer"])
            minute = format_ordinal_minute(row["minute"])

            if scorer not in scorer_minutes:
                scorer_minutes[scorer] = []

            scorer_minutes[scorer].append(minute)

        details = []

        for scorer, minutes in scorer_minutes.items():

            # One goal
            if len(minutes) == 1:
                details.append(
                    f"{scorer} scored in the {minutes[0]} minute"
                )

            # Two goals
            elif len(minutes) == 2:
                details.append(
                    f"{scorer} scored in the "
                    f"{minutes[0]} and {minutes[1]} minutes"
                )

            # Three or more goals
            else:
                minute_text = (
                    ", ".join(minutes[:-1])
                    + f", and {minutes[-1]}"
                )

                details.append(
                    f"{scorer} scored in the "
                    f"{minute_text} minutes"
                )

        return "; ".join(details) + "."

    # -------------------------------------------------
    # OWN GOAL STATUS
    # -------------------------------------------------
    elif intent == "own_goal_status":
        details = []

        for _, row in goal_events.iterrows():
            status = (
                "was"
                if bool(row["own_goal"])
                else "was not"
            )

            details.append(
                f"{row['scorer']}'s goal "
                f"{status} an own goal"
            )

        return "; ".join(details) + "."

    # -------------------------------------------------
    # PENALTY STATUS
    # -------------------------------------------------
    elif intent == "penalty_status":
        details = []

        for _, row in goal_events.iterrows():
            status = (
                "was"
                if bool(row["penalty"])
                else "was not"
            )

            details.append(
                f"{row['scorer']}'s goal "
                f"{status} a penalty"
            )

        return "; ".join(details) + "."

    # -------------------------------------------------
    # FALLBACK
    # -------------------------------------------------
    return (
        "Answer generation for this intent "
        "is not available yet."
    )

In [169]:
# Cell 46 - Test scorer answer generation

question = "Which player scored for Paraguay against Chile?"

intent = predict_intent(question)

answer = generate_goalscorer_answer(
    intent,
    selected_goal_events
)

print("Question:", question)
print("Intent:", intent)
print("Answer:", answer)

Question: Which player scored for Paraguay against Chile?
Intent: scorer
Answer: The scorers were Braian Samudio, Miguel Almirón in the Chile vs Paraguay match on 2021-06-24.


# Cell 47 - Interactive goalscorer chatbot

def ask_goalscorer_question(question):
    intent = predict_intent(question)

    #print(f"\nPredicted intent: {intent}")

    if intent not in GOALSCORER_INTENTS:
        print(
            "\nAssistant: This question does not belong to "
            "the goalscorer section."
        )
        return

    # Initial goal-event candidates
    goal_candidates = find_goalscorer_candidates(question)

    if goal_candidates.empty:
        print(
            "\nAssistant: I couldn't find a goal record "
            "matching your question."
        )
        return

    # Convert goal events into unique possible matches
    unique_matches = get_unique_goal_matches(goal_candidates)

    enriched_matches = enrich_goal_matches(unique_matches)

    print("Possible matches found:", len(enriched_matches))

    # Resolve MATCH ambiguity
    while len(enriched_matches) > 1:

        clarification = generate_goal_match_clarification(
            enriched_matches
        )

        print(f"\nAssistant: {clarification}")

        reply = input("\nYou: ").strip()

        if not reply:
            print(
                "\nAssistant: Please provide another detail "
                "about the match."
            )
            continue

        new_matches = apply_goal_match_clarification(
            enriched_matches,
            reply
        )

        if len(new_matches) == len(enriched_matches):
            print(
                "\nAssistant: That detail didn't narrow the "
                "possible matches. Please try another detail."
            )
            continue

        if len(new_matches) == 0:
            print(
                "\nAssistant: I couldn't find a remaining match "
                "with that detail. Please try something else."
            )
            continue

        enriched_matches = new_matches

        print(
            f"\nRemaining possible matches: "
            f"{len(enriched_matches)}"
        )

    # Exactly one match identified
    selected_match = enriched_matches.iloc[0]
    selected_key = selected_match["match_key"]

    # Return ALL relevant goal events from that match
    selected_goal_events = goal_candidates[
        goal_candidates["match_key"] == selected_key
    ].copy()

    answer = generate_goalscorer_answer(
        intent,
        selected_goal_events
    )

    print(f"\nAssistant: {answer}")

In [170]:
# Cell 49 - Test goal_minute intent

question = "At what minute did Christian Eriksen find the net against Slovenia?"

print("Intent:", predict_intent(question))

goal_candidates = find_goalscorer_candidates(question)

print("Goal records found:", len(goal_candidates))

display(
    goal_candidates[
        [
            "date",
            "home_team",
            "away_team",
            "team",
            "scorer",
            "minute",
            "own_goal",
            "penalty"
        ]
    ].head(20)
)

Intent: goal_minute
Goal records found: 1


,date,home_team,away_team,team,scorer,minute,own_goal,penalty
44962,2024-06-16,Slovenia,Denmark,Denmark,Christian Eriksen,17,False,False


In [171]:
# Cell 50 - Test penalty_status intent

question = "Did Georgios Samaras convert a penalty against Germany on 2012-06-22?"

print("Intent:", predict_intent(question))

goal_candidates = find_goalscorer_candidates(question)

print("Goal records found:", len(goal_candidates))

display(
    goal_candidates[
        [
            "date",
            "home_team",
            "away_team",
            "team",
            "scorer",
            "minute",
            "own_goal",
            "penalty"
        ]
    ]
)

Intent: penalty_status
Goal records found: 1


,date,home_team,away_team,team,scorer,minute,own_goal,penalty
32014,2012-06-22,Germany,Greece,Greece,Georgios Samaras,55,False,False


In [172]:
# Cell 51 - Test own_goal_status intent

question = "Did Jordi Alba score an own goal against Italy?"

print("Intent:", predict_intent(question))

goal_candidates = find_goalscorer_candidates(question)

print("Goal records found:", len(goal_candidates))

display(
    goal_candidates[
        [
            "date",
            "home_team",
            "away_team",
            "team",
            "scorer",
            "minute",
            "own_goal",
            "penalty"
        ]
    ].head(20)
)

Intent: own_goal_status
Goal records found: 1


,date,home_team,away_team,team,scorer,minute,own_goal,penalty
32025,2012-07-01,Spain,Italy,Spain,Jordi Alba,41,False,False


In [173]:
# Cell 52 - Final interactive goalscorer chatbot
# Clean chatbot output + optional debug mode

def ask_goalscorer_question(question):
    intent = predict_intent(question)

    if DEBUG:
        print(f"\nPredicted intent: {intent}")

    if intent not in GOALSCORER_INTENTS:
        print(
            "\nAssistant: This question does not belong to "
            "the goalscorer section."
        )
        return

    # Initial goal-event candidates
    goal_candidates = find_goalscorer_candidates(question)

    if goal_candidates.empty:
        print(
            "\nAssistant: I couldn't find a goal record "
            "matching your question."
        )
        return

    # Convert goal events into unique matches
    unique_matches = get_unique_goal_matches(goal_candidates)
    enriched_matches = enrich_goal_matches(unique_matches)

    if DEBUG:
        print(
            "Possible matches found:",
            len(enriched_matches)
        )

    # -------------------------------------------------
    # Resolve match ambiguity
    # -------------------------------------------------
    while len(enriched_matches) > 1:

        clarification = generate_goal_match_clarification(
            enriched_matches
        )

        print(f"\nAssistant: {clarification}")

        reply = input("\nYou: ").strip()

        if not reply:
            print(
                "\nAssistant: Please provide another detail "
                "about the match."
            )
            continue

        new_matches = apply_goal_match_clarification(
            enriched_matches,
            reply
        )

        if len(new_matches) == len(enriched_matches):
            print(
                "\nAssistant: That detail didn't narrow the "
                "possible matches. Please try another detail."
            )
            continue

        if len(new_matches) == 0:
            print(
                "\nAssistant: I couldn't find a remaining match "
                "with that detail. Please try something else."
            )
            continue

        enriched_matches = new_matches

        if DEBUG:
            print(
                f"\nRemaining possible matches: "
                f"{len(enriched_matches)}"
            )

    # -------------------------------------------------
    # Exactly one match remains
    # -------------------------------------------------
    selected_match = enriched_matches.iloc[0]
    selected_key = selected_match["match_key"]

    # Keep ALL relevant goals from the selected match
    selected_goal_events = goal_candidates[
        goal_candidates["match_key"] == selected_key
    ].copy()

    answer = generate_goalscorer_answer(
        intent,
        selected_goal_events
    )

    print(f"\nAssistant: {answer}")

In [174]:
# Cell 48 - Full goalscorer conversation test

question = "Which player scored for Paraguay against Chile?"

ask_goalscorer_question(question)


Assistant: I found matches across 29 different years. Do you remember roughly which year it was?

You: 2025

Assistant: Omar Alderete scored in the Paraguay vs Chile match on 2025-03-20.


In [175]:
ask_goalscorer_question(
    "At what minute did Christian Eriksen find the net against Slovenia?"
)


Assistant: Christian Eriksen scored in the 17th minute.


In [176]:
ask_goalscorer_question(
    "Did Georgios Samaras convert a penalty against Germany on 2012-06-22?"
)


Assistant: Georgios Samaras's goal was not a penalty.


In [177]:
ask_goalscorer_question(
    "Did Jordi Alba score an own goal against Italy?"
)


Assistant: Jordi Alba's goal was not an own goal.


In [178]:
# Cell 53 - Prepare shootout data

SHOOTOUT_INTENTS = {
    "shootout_winner",
    "first_shooter"
}

shootouts_with_key = add_match_key(shootouts_df)

print("Shootout records:", len(shootouts_with_key))
print("\nColumns:")
print(shootouts_with_key.columns.tolist())

Shootout records: 682

Columns:
['date', 'home_team', 'away_team', 'winner', 'first_shooter', 'match_key']


In [179]:
# Cell 54 - Extract entities for shootout questions

def extract_shootout_entities(question):
    question_normalized = normalize_text(question)

    found_teams = [
        team
        for team in all_teams
        if normalize_text(team) in question_normalized
    ]

    year_match = re.search(
        r"\b(18|19|20)\d{2}\b",
        question
    )

    year = year_match.group() if year_match else None

    return {
        "teams": found_teams,
        "year": year
    }

In [180]:
# Cell 55 - Find shootout candidates

def find_shootout_candidates(question):
    entities = extract_shootout_entities(question)

    candidates = shootouts_with_key.copy()

    teams = entities["teams"]
    year = entities["year"]

    # Two teams mentioned
    if len(teams) >= 2:
        team1, team2 = teams[:2]

        candidates = candidates[
            (
                (candidates["home_team"] == team1)
                & (candidates["away_team"] == team2)
            )
            |
            (
                (candidates["home_team"] == team2)
                & (candidates["away_team"] == team1)
            )
        ]

    # Year
    if year:
        candidates = candidates[
            candidates["date"]
            .astype(str)
            .str.startswith(year)
        ]

    return candidates

In [181]:
# Cell 56 - Test shootout retrieval

question = (
    "Who took the first penalty in the shootout "
    "between Uruguay and Ghana?"
)

intent = predict_intent(question)

shootout_candidates = find_shootout_candidates(
    question
)

print("Predicted intent:", intent)
print(
    "Possible shootout records:",
    len(shootout_candidates)
)

display(
    shootout_candidates[
        [
            "date",
            "home_team",
            "away_team",
            "winner",
            "first_shooter"
        ]
    ]
)

Predicted intent: first_shooter
Possible shootout records: 1


,date,home_team,away_team,winner,first_shooter
409,2010-07-02,Uruguay,Ghana,Uruguay,Uruguay


In [182]:
# Cell 57 - Enrich shootout records with results data

def enrich_shootout_matches(shootout_candidates):
    return shootout_candidates.merge(
        results_with_key[
            [
                "match_key",
                "home_score",
                "away_score",
                "tournament",
                "city",
                "country",
                "neutral"
            ]
        ],
        on="match_key",
        how="left"
    )

In [183]:
enriched_shootouts = enrich_shootout_matches(
    shootout_candidates
)

display(
    enriched_shootouts[
        [
            "date",
            "home_team",
            "away_team",
            "winner",
            "first_shooter",
            "tournament",
            "city",
            "country"
        ]
    ]
)

,date,home_team,away_team,winner,first_shooter,tournament,city,country
0,2010-07-02,Uruguay,Ghana,Uruguay,Uruguay,FIFA World Cup,Johannesburg,South Africa


In [184]:
# Cell 58 - Generate answers for shootout intents

def generate_shootout_answer(intent, row):
    home_team = row["home_team"]
    away_team = row["away_team"]
    date = row["date"]

    if intent == "shootout_winner":
        winner = row["winner"]

        return (
            f"{winner} won the penalty shootout "
            f"between {home_team} and {away_team} on {date}."
        )

    elif intent == "first_shooter":
        first_shooter = row["first_shooter"]

        if pd.isna(first_shooter) or str(first_shooter).strip().lower() == "unknown":
            return (
                f"The first shooter is not recorded for the "
                f"{home_team} vs {away_team} shootout on {date}."
            )

        return (
            f"{first_shooter} took the first penalty in the "
            f"{home_team} vs {away_team} shootout on {date}."
        )

    return "Answer generation for this shootout intent is not available yet."

In [185]:
question = (
    "Who took the first penalty in the shootout "
    "between Uruguay and Ghana?"
)

intent = predict_intent(question)

shootout_candidates = find_shootout_candidates(question)

enriched_shootouts = enrich_shootout_matches(
    shootout_candidates
)

answer = generate_shootout_answer(
    intent,
    enriched_shootouts.iloc[0]
)

print("Question:", question)
print("Intent:", intent)
print("Answer:", answer)

Question: Who took the first penalty in the shootout between Uruguay and Ghana?
Intent: first_shooter
Answer: Uruguay took the first penalty in the Uruguay vs Ghana shootout on 2010-07-02.


In [186]:
question = (
    "Who won the penalty shootout between Uruguay and Ghana?"
)

intent = predict_intent(question)

shootout_candidates = find_shootout_candidates(question)

enriched_shootouts = enrich_shootout_matches(
    shootout_candidates
)

answer = generate_shootout_answer(
    intent,
    enriched_shootouts.iloc[0]
)

print("Question:", question)
print("Intent:", intent)
print("Answer:", answer)

Question: Who won the penalty shootout between Uruguay and Ghana?
Intent: shootout_winner
Answer: Uruguay won the penalty shootout between Uruguay and Ghana on 2010-07-02.


In [187]:
# Cell 60 - Find team pairs with multiple penalty shootouts

shootout_pairs = shootouts_with_key.copy()

shootout_pairs["team_pair"] = shootout_pairs.apply(
    lambda row: tuple(
        sorted([row["home_team"], row["away_team"]])
    ),
    axis=1
)

repeated_shootout_pairs = (
    shootout_pairs
    .groupby("team_pair")
    .size()
    .sort_values(ascending=False)
)

repeated_shootout_pairs = repeated_shootout_pairs[
    repeated_shootout_pairs > 1
]

print(
    "Team pairs with multiple shootouts:",
    len(repeated_shootout_pairs)
)

print("\nTop repeated shootout matchups:\n")
print(repeated_shootout_pairs.head(20))

Team pairs with multiple shootouts: 100

Top repeated shootout matchups:

team_pair
(Indonesia, Thailand)       5
(Guinea, Mali)              5
(Guernsey, Jersey)          5
(Kenya, Uganda)             5
(Brazil, Uruguay)           4
(Botswana, South Africa)    4
(Costa Rica, Honduras)      4
(Japan, South Korea)        4
(Zambia, Zimbabwe)          4
(Argentina, Brazil)         4
(Malawi, South Africa)      4
(Egypt, Ivory Coast)        3
(Lesotho, Mozambique)       3
(Malawi, Zambia)            3
(Argentina, Netherlands)    3
(Argentina, Colombia)       3
(Malaysia, Thailand)        3
(Panama, United States)     3
(Costa Rica, Mexico)        3
(Nigeria, Tunisia)          3
dtype: int64


In [188]:
# Cell 61 - Inspect the most repeated shootout matchup

test_pair = repeated_shootout_pairs.index[0]

team1, team2 = test_pair

print("Testing:", team1, "vs", team2)

test_shootouts = shootouts_with_key[
    (
        (shootouts_with_key["home_team"] == team1)
        & (shootouts_with_key["away_team"] == team2)
    )
    |
    (
        (shootouts_with_key["home_team"] == team2)
        & (shootouts_with_key["away_team"] == team1)
    )
]

enriched_test_shootouts = enrich_shootout_matches(
    test_shootouts
)

display(
    enriched_test_shootouts[
        [
            "date",
            "home_team",
            "away_team",
            "winner",
            "first_shooter",
            "tournament",
            "city",
            "country"
        ]
    ]
)

Testing: Indonesia vs Thailand


,date,home_team,away_team,winner,first_shooter,tournament,city,country
0,1979-09-29,Indonesia,Thailand,Indonesia,Unknown,Southeast Asian Games,Jakarta,Indonesia
1,1989-08-30,Indonesia,Thailand,Indonesia,Unknown,Southeast Asian Games,Kuala Lumpur,Malaysia
2,1991-12-04,Indonesia,Thailand,Indonesia,Unknown,Southeast Asian Games,Manila,Philippines
3,1997-10-18,Indonesia,Thailand,Thailand,Unknown,Southeast Asian Games,Jakarta,Indonesia
4,1998-09-05,Indonesia,Thailand,Indonesia,Unknown,AFF Championship,Ho Chi Minh City,Vietnam


In [189]:
# Cell 62 - Smart clarification for shootout candidates

def generate_shootout_clarification(enriched_shootouts):
    if len(enriched_shootouts) == 0:
        return "I couldn't find a matching shootout."

    if len(enriched_shootouts) == 1:
        return None

    return generate_smart_clarification(enriched_shootouts)

In [190]:
# Cell 63 - Apply clarification to shootout candidates

def apply_shootout_clarification(enriched_shootouts, reply):
    return apply_smart_clarification(
        enriched_shootouts,
        reply
    )

In [191]:
# Cell 64 - Interactive shootout chatbot

def ask_shootout_question(question):
    intent = predict_intent(question)

    #print(f"\nPredicted intent: {intent}")

    if intent not in SHOOTOUT_INTENTS:
        print(
            "\nAssistant: This question does not belong to "
            "the shootout section."
        )
        return

    shootout_candidates = find_shootout_candidates(question)

    if shootout_candidates.empty:
        print(
            "\nAssistant: I couldn't find a penalty shootout "
            "matching your question."
        )
        return

    enriched_shootouts = enrich_shootout_matches(
        shootout_candidates
    )

    print(
        "Possible shootout records:",
        len(enriched_shootouts)
    )

    while len(enriched_shootouts) > 1:
        clarification = generate_shootout_clarification(
            enriched_shootouts
        )

        print(f"\nAssistant: {clarification}")

        reply = input("\nYou: ").strip()

        if not reply:
            print(
                "\nAssistant: Please provide another detail "
                "about the shootout."
            )
            continue

        new_shootouts = apply_shootout_clarification(
            enriched_shootouts,
            reply
        )

        if len(new_shootouts) == len(enriched_shootouts):
            print(
                "\nAssistant: That detail didn't narrow the "
                "possible shootouts. Please try another detail."
            )
            continue

        if len(new_shootouts) == 0:
            print(
                "\nAssistant: I couldn't match that detail to "
                "the remaining shootouts."
            )
            continue

        enriched_shootouts = new_shootouts

        print(
            f"\nRemaining shootouts: "
            f"{len(enriched_shootouts)}"
        )

    selected_shootout = enriched_shootouts.iloc[0]

    answer = generate_shootout_answer(
        intent,
        selected_shootout
    )

    print(f"\nAssistant: {answer}")

In [192]:
  # Cell 65 - Full shootout ambiguity test

question = (
    "Who won the penalty shootout between "
    "Indonesia and Thailand?"
)

ask_shootout_question(question)

Possible shootout records: 5

Assistant: I found matches from multiple years. Do you remember the year? Options: 1979, 1989, 1991, 1997, 1998

You: 1997

Remaining shootouts: 1

Assistant: Thailand won the penalty shootout between Indonesia and Thailand on 1997-10-18.


In [193]:
# Cell 66 - Inspect exact major tournament names in the dataset

keywords = [
    "World Cup",
    "Euro",
    "Copa",
    "African",
    "Asian Cup",
    "Gold Cup",
    "OFC",
    "Nations League"
]

major_tournament_names = sorted(
    tournament
    for tournament in results_df["tournament"]
        .dropna()
        .astype(str)
        .unique()
    if any(
        keyword.lower() in tournament.lower()
        for keyword in keywords
    )
)

print("Possible major tournament names:\n")

for tournament in major_tournament_names:
    print("-", tournament)

Possible major tournament names:

- AFC Asian Cup
- AFC Asian Cup qualification
- African Cup of Nations
- African Cup of Nations qualification
- African Friendship Games
- All-African Games
- CONCACAF Nations League
- CONCACAF Nations League qualification
- CONIFA European Football Cup
- CONIFA World Cup qualification
- Central European International Cup
- Copa América
- Copa América qualification
- Copa Artigas
- Copa Bernardo O'Higgins
- Copa Carlos Dittborn
- Copa Chevallier Boutell
- Copa Confraternidad
- Copa Félix Bogado
- Copa Juan Pinto Durán
- Copa Lipton
- Copa Newton
- Copa Oswaldo Cruz
- Copa Paz del Chaco
- Copa Premio Honor Argentino
- Copa Premio Honor Uruguayo
- Copa Ramón Castilla
- Copa Rio Branco
- Copa Roca
- Copa del Pacífico
- FIFA World Cup
- FIFA World Cup qualification
- Gold Cup
- Gold Cup qualification
- Morocco, Capital of African Football
- UEFA Euro
- UEFA Euro qualification
- UEFA Nations League
- Viva World Cup
- West African Cup


In [194]:
# Cell 67 - Supported major international tournaments

SUPPORTED_MAJOR_TOURNAMENTS = {
    # FIFA World Cup
    "world cup": "FIFA World Cup",
    "worldcup": "FIFA World Cup",
    "worldcup": "FIFA World Cup",
    "fifa worldcup": "FIFA World Cup",

    # UEFA European Championship
    "euro": "UEFA Euro",
    "euros": "UEFA Euro",
    "european championship": "UEFA Euro",
    "uefa euro": "UEFA Euro",

    # Copa América
    "copa america": "Copa América",
    "copa américa": "Copa América",

    # Africa Cup of Nations
    "afcon": "African Cup of Nations",
    "africa cup of nations": "African Cup of Nations",
    "african cup of nations": "African Cup of Nations",

    # AFC Asian Cup
    "asian cup": "AFC Asian Cup",
    "afc asian cup": "AFC Asian Cup",

    # CONCACAF Gold Cup
    "gold cup": "Gold Cup",
    "concacaf gold cup": "Gold Cup",

    # OFC / Oceania Nations Cup
    "ofc nations cup": "Oceania Nations Cup",
    "oceania nations cup": "Oceania Nations Cup",

    # UEFA Nations League
    "uefa nations league": "UEFA Nations League",

    # CONCACAF Nations League
    "concacaf nations league": "CONCACAF Nations League"
}

print("Supported major tournament aliases:\n")

for alias, tournament in SUPPORTED_MAJOR_TOURNAMENTS.items():
    print(f"{alias} -> {tournament}")

Supported major tournament aliases:

world cup -> FIFA World Cup
worldcup -> FIFA World Cup
fifa worldcup -> FIFA World Cup
euro -> UEFA Euro
euros -> UEFA Euro
european championship -> UEFA Euro
uefa euro -> UEFA Euro
copa america -> Copa América
copa américa -> Copa América
afcon -> African Cup of Nations
africa cup of nations -> African Cup of Nations
african cup of nations -> African Cup of Nations
asian cup -> AFC Asian Cup
afc asian cup -> AFC Asian Cup
gold cup -> Gold Cup
concacaf gold cup -> Gold Cup
ofc nations cup -> Oceania Nations Cup
oceania nations cup -> Oceania Nations Cup
uefa nations league -> UEFA Nations League
concacaf nations league -> CONCACAF Nations League


In [195]:
ofc_search = sorted(
    tournament
    for tournament in results_df["tournament"]
        .dropna()
        .astype(str)
        .unique()
    if (
        "ofc" in tournament.lower()
        or "oceania" in tournament.lower()
        or "nations cup" in tournament.lower()
    )
)

print("Possible Oceania tournament names:\n")

for tournament in ofc_search:
    print("-", tournament)

Possible Oceania tournament names:

- CAFA Nations Cup
- Mauritius Four Nations Cup
- Nations Cup
- Oceania Nations Cup
- Oceania Nations Cup qualification
- Three Nations Cup
- Tri-Nations Cup


In [196]:
# Cell 68 - Detect major tournament from user question

def detect_major_tournament(question):
    question_normalized = normalize_text(question)

    # Longest aliases first so specific names such as
    # "concacaf gold cup" are checked before "gold cup"
    aliases = sorted(
        SUPPORTED_MAJOR_TOURNAMENTS.keys(),
        key=len,
        reverse=True
    )

    for alias in aliases:
        if normalize_text(alias) in question_normalized:
            return SUPPORTED_MAJOR_TOURNAMENTS[alias]

    return None

In [197]:
test_questions = [
    "Who won the 2022 World Cup final?",
    "Who won the Euro 2024 final?",
    "Who won the 2024 Copa America final?",
    "Who won the 2023 AFCON final?",
    "Who won the 2023 Asian Cup final?",
    "Who won the 2023 Gold Cup final?",
    "Who won the 2024 OFC Nations Cup final?",
    "Who won the 2025 UEFA Nations League final?",
    "Who won the 2025 CONCACAF Nations League final?"
]

for question in test_questions:
    print(
        question,
        "->",
        detect_major_tournament(question)
    )

Who won the 2022 World Cup final? -> FIFA World Cup
Who won the Euro 2024 final? -> UEFA Euro
Who won the 2024 Copa America final? -> Copa América
Who won the 2023 AFCON final? -> African Cup of Nations
Who won the 2023 Asian Cup final? -> AFC Asian Cup
Who won the 2023 Gold Cup final? -> Gold Cup
Who won the 2024 OFC Nations Cup final? -> Oceania Nations Cup
Who won the 2025 UEFA Nations League final? -> UEFA Nations League
Who won the 2025 CONCACAF Nations League final? -> CONCACAF Nations League


In [198]:
def find_tournament_final(question):
    tournament = detect_major_tournament(question)

    if tournament is None:
        return None, "I couldn't identify a supported major tournament."

    # Extract edition year
    year_match = re.search(r"\b(18|19|20)\d{2}\b", question)

    if not year_match:
        return None, "I couldn't identify the tournament year."

    edition_year = year_match.group()

    # Handle editions played in a different calendar year
    actual_year = TOURNAMENT_EDITION_YEAR_OVERRIDES.get(
        (tournament, edition_year),
        edition_year
    )

    # Filter matches from the correct tournament edition
    edition_matches = results_df[
        (results_df["tournament"] == tournament)
        &
        (
            results_df["date"]
            .astype(str)
            .str.startswith(actual_year)
        )
    ].copy()

    if edition_matches.empty:
        return None, (
            f"I couldn't find {tournament} matches "
            f"for the {edition_year} edition."
        )

    # Sort chronologically
    edition_matches = edition_matches.sort_values("date")

    # Latest match in the edition = inferred final
    final_match = edition_matches.iloc[-1]

    return final_match, None

In [199]:
# Cell 82 - Major tournament edition-year exceptions

TOURNAMENT_EDITION_YEAR_OVERRIDES = {
    # Tournament edition -> actual calendar year played
    ("African Cup of Nations", "2023"): "2024",
    ("African Cup of Nations", "2021"): "2022",

    ("AFC Asian Cup", "2023"): "2024",

    # UEFA Euro 2020 was played in 2021
    ("UEFA Euro", "2020"): "2021"
}

In [200]:
# Cell 70 - Test major tournament final detection

test_questions = [
    "Who won the 2022 World Cup final?",
    "Who won the Euro 2024 final?",
    "Who won the 2024 Copa America final?"
]

for question in test_questions:
    final_match, error = find_tournament_final(question)

    print("\nQuestion:", question)

    if error:
        print("Error:", error)
    else:
        print(
            final_match[
                [
                    "date",
                    "home_team",
                    "away_team",
                    "home_score",
                    "away_score",
                    "tournament"
                ]
            ]
        )


Question: Who won the 2022 World Cup final?
date              2022-12-18
home_team          Argentina
away_team             France
home_score                 3
away_score                 3
tournament    FIFA World Cup
Name: 45786, dtype: object

Question: Who won the Euro 2024 final?
date          2024-07-14
home_team          Spain
away_team        England
home_score             2
away_score             1
tournament     UEFA Euro
Name: 47470, dtype: object

Question: Who won the 2024 Copa America final?
date            2024-07-14
home_team        Argentina
away_team         Colombia
home_score               1
away_score               0
tournament    Copa América
Name: 47471, dtype: object


In [201]:
# Cell 71 - Resolve winner of an inferred tournament final

def resolve_final_winner(final_match):
    home_team = final_match["home_team"]
    away_team = final_match["away_team"]
    home_score = final_match["home_score"]
    away_score = final_match["away_score"]
    date = str(final_match["date"])
    tournament = final_match["tournament"]

    # Normal win
    if home_score > away_score:
        return (
            f"{home_team} won the {tournament} final "
            f"{home_score}-{away_score} against {away_team}."
        )

    if away_score > home_score:
        return (
            f"{away_team} won the {tournament} final "
            f"{away_score}-{home_score} against {home_team}."
        )

    # Tied score -> check shootout dataset
    shootout_match = shootouts_df[
        (shootouts_df["date"].astype(str) == date)
        &
        (
            (
                (shootouts_df["home_team"] == home_team)
                &
                (shootouts_df["away_team"] == away_team)
            )
            |
            (
                (shootouts_df["home_team"] == away_team)
                &
                (shootouts_df["away_team"] == home_team)
            )
        )
    ]

    if not shootout_match.empty:
        shootout_winner = shootout_match.iloc[0]["winner"]

        return (
            f"{shootout_winner} won the {tournament} final on penalties "
            f"after a {home_score}-{away_score} draw between "
            f"{home_team} and {away_team}."
        )

    # Genuine draw / no shootout record
    return (
        f"The {tournament} match between {home_team} and {away_team} "
        f"ended {home_score}-{away_score}, and no shootout record was found."
    )

In [202]:
# Cell 72 - Test final winner resolution

test_questions = [
    "Who won the 2022 World Cup final?",
    "Who won the Euro 2024 final?",
    "Who won the 2024 Copa America final?"
]

for question in test_questions:
    final_match, error = find_tournament_final(question)

    print("\nQuestion:", question)

    if error:
        print("Assistant:", error)
    else:
        print("Assistant:", resolve_final_winner(final_match))


Question: Who won the 2022 World Cup final?
Assistant: Argentina won the FIFA World Cup final on penalties after a 3-3 draw between Argentina and France.

Question: Who won the Euro 2024 final?
Assistant: Spain won the UEFA Euro final 2-1 against England.

Question: Who won the 2024 Copa America final?
Assistant: Argentina won the Copa América final 1-0 against Colombia.


In [203]:
# Detect tournament champion / winner questions

def is_tournament_winner_question(question):
    q = normalize_text(question)

    winner_phrases = [
        "who won",
        "winner",
        "champion",
        "champions",
        "who became champion",
        "who were champions"
    ]

    return any(
        phrase in q
        for phrase in winner_phrases
    )

In [204]:
# Cell 84 - Detect ambiguous generic Nations League references

def detect_generic_nations_league(question):
    q = normalize_text(question)

    if "nations league" not in q:
        return False

    # Already specific, so not ambiguous
    if "uefa nations league" in q:
        return False

    if "concacaf nations league" in q:
        return False

    return True

In [205]:
# Cell 84 - Detect ambiguous generic Nations League references

def detect_generic_nations_league(question):
    q = normalize_text(question)

    if "nations league" not in q:
        return False

    # Already specific, so not ambiguous
    if "uefa nations league" in q:
        return False

    if "concacaf nations league" in q:
        return False

    return True

In [206]:
# Infer whether a generic "Nations League" question
# refers to UEFA or CONCACAF based on the teams mentioned

UEFA_NATIONS_LEAGUE_TEAMS = set(
    results_df[
        results_df["tournament"] == "UEFA Nations League"
    ]["home_team"].dropna().tolist()
    +
    results_df[
        results_df["tournament"] == "UEFA Nations League"
    ]["away_team"].dropna().tolist()
)

CONCACAF_NATIONS_LEAGUE_TEAMS = set(
    results_df[
        results_df["tournament"] == "CONCACAF Nations League"
    ]["home_team"].dropna().tolist()
    +
    results_df[
        results_df["tournament"] == "CONCACAF Nations League"
    ]["away_team"].dropna().tolist()
)


def infer_nations_league_from_teams(question):
    # Only apply this to generic "Nations League" wording
    if not detect_generic_nations_league(question):
        return None

    entities = extract_basic_entities(question)
    teams = entities["teams"]

    if not teams:
        return None

    uefa_matches = [
        team
        for team in teams
        if team in UEFA_NATIONS_LEAGUE_TEAMS
    ]

    concacaf_matches = [
        team
        for team in teams
        if team in CONCACAF_NATIONS_LEAGUE_TEAMS
    ]

    # Teams clearly point to UEFA
    if uefa_matches and not concacaf_matches:
        return "UEFA Nations League"

    # Teams clearly point to CONCACAF
    if concacaf_matches and not uefa_matches:
        return "CONCACAF Nations League"

    # Still ambiguous
    return None

In [207]:
# Cell 86 - Resolve generic Nations League references

def resolve_nations_league_reference(question):
    # Not a generic Nations League question
    if not detect_generic_nations_league(question):
        return question

    # Try to infer competition from mentioned teams
    inferred = infer_nations_league_from_teams(question)

    if inferred:
        print(
            f"\nAssistant: I identified this as the "
            f"{inferred} based on the teams mentioned."
        )

        # Replace generic wording with exact tournament name
        return re.sub(
            r"\bnations league\b",
            inferred,
            question,
            flags=re.IGNORECASE
        )

    # Cannot infer safely -> ask user
    print(
        "\nAssistant: Do you mean the UEFA Nations League "
        "or the CONCACAF Nations League?"
    )

    while True:
        reply = input("\nYou: ").strip()
        reply_normalized = normalize_text(reply)

        if (
            "uefa" in reply_normalized
            or "europe" in reply_normalized
            or "european" in reply_normalized
        ):
            tournament = "UEFA Nations League"
            break

        if (
            "concacaf" in reply_normalized
            or "north america" in reply_normalized
            or "central america" in reply_normalized
        ):
            tournament = "CONCACAF Nations League"
            break

        print(
            "\nAssistant: Please specify either UEFA "
            "or CONCACAF."
        )

    return re.sub(
        r"\bnations league\b",
        tournament,
        question,
        flags=re.IGNORECASE
    )

In [208]:
# Cell 87 - Unified Random Forest football QA router
# Clean chatbot output + optional debug mode
# Includes Nations League ambiguity handling

def football_qa(question):

    # -------------------------------------------------
    # 1. Resolve generic Nations League wording
    # -------------------------------------------------
    question = resolve_nations_league_reference(question)

    # -------------------------------------------------
    # 2. Predict intent using Random Forest
    # -------------------------------------------------
    intent = predict_intent(question)

    if DEBUG:
        print(f"\nPredicted intent: {intent}")

    question_normalized = normalize_text(question)
    tournament = detect_major_tournament(question)

    # -------------------------------------------------
    # 3. Major tournament final questions
    # -------------------------------------------------
    if (
        intent == "match_winner"
        and tournament is not None
        and (
            "final" in question_normalized
            or is_tournament_winner_question(question)
        )
    ):
        final_match, error = find_tournament_final(question)

        if error is None:
            print(
                "\nAssistant:",
                resolve_final_winner(final_match)
            )
            return

    # -------------------------------------------------
    # 4. Results intents
    # -------------------------------------------------
    if intent in RESULTS_INTENTS:
        ask_results_question(question)
        return

    # -------------------------------------------------
    # 5. Goalscorer intents
    # -------------------------------------------------
    if intent in GOALSCORER_INTENTS:
        ask_goalscorer_question(question)
        return

    # -------------------------------------------------
    # 6. Shootout intents
    # -------------------------------------------------
    if intent in SHOOTOUT_INTENTS:
        ask_shootout_question(question)
        return

    # -------------------------------------------------
    # 7. Unsupported intent
    # -------------------------------------------------
    print(
        "\nAssistant: I understood the question, "
        "but I don't currently have an answer handler "
        "for this type of question."
    )

In [209]:
checks = [
    "resolve_nations_league_reference",
    "is_tournament_winner_question",
    "detect_major_tournament",
    "find_tournament_final",
    "resolve_final_winner",
    "football_qa"
]

for name in checks:
    print(name, "->", name in globals())

resolve_nations_league_reference -> True
is_tournament_winner_question -> True
detect_major_tournament -> True
find_tournament_final -> True
resolve_final_winner -> True
football_qa -> True


In [210]:
question = "who won the world cup in 2010"

print("Tournament detected:", detect_major_tournament(question))
print("Winner question:", is_tournament_winner_question(question))
print("Predicted intent:", predict_intent(question))

Tournament detected: FIFA World Cup
Winner question: True
Predicted intent: match_winner


In [211]:
import inspect

print(inspect.getsource(football_qa))

def football_qa(question):

    # -------------------------------------------------
    # 1. Resolve generic Nations League wording
    # -------------------------------------------------
    question = resolve_nations_league_reference(question)

    # -------------------------------------------------
    # 2. Predict intent using Random Forest
    # -------------------------------------------------
    intent = predict_intent(question)

    if DEBUG:
        print(f"\nPredicted intent: {intent}")

    question_normalized = normalize_text(question)
    tournament = detect_major_tournament(question)

    # -------------------------------------------------
    # 3. Major tournament final questions
    # -------------------------------------------------
    if (
        intent == "match_winner"
        and tournament is not None
        and (
            "final" in question_normalized
            or is_tournament_winner_question(question)
        )
    ):
        final_match, error = find

In [212]:
print(inspect.getsource(find_tournament_final))

def find_tournament_final(question):
    tournament = detect_major_tournament(question)

    if tournament is None:
        return None, "I couldn't identify a supported major tournament."

    # Extract edition year
    year_match = re.search(r"\b(18|19|20)\d{2}\b", question)

    if not year_match:
        return None, "I couldn't identify the tournament year."

    edition_year = year_match.group()

    # Handle editions played in a different calendar year
    actual_year = TOURNAMENT_EDITION_YEAR_OVERRIDES.get(
        (tournament, edition_year),
        edition_year
    )

    # Filter matches from the correct tournament edition
    edition_matches = results_df[
        (results_df["tournament"] == tournament)
        &
        (
            results_df["date"]
            .astype(str)
            .str.startswith(actual_year)
        )
    ].copy()

    if edition_matches.empty:
        return None, (
            f"I couldn't find {tournament} matches "
            f"for the {editi

In [213]:
import inspect

print(inspect.getsource(apply_smart_clarification))

def apply_smart_clarification(candidates, reply):
    reply_normalized = normalize_text(reply)
    filtered = candidates.copy()

    # -------------------------------------------------
    # 0. EXACT DATE - MUST COME BEFORE YEAR
    # -------------------------------------------------
    exact_date = extract_exact_date(reply)

    if exact_date:
        candidate_dates = pd.to_datetime(
            filtered["date"],
            errors="coerce"
        ).dt.strftime("%Y-%m-%d")

        date_filtered = filtered[
            candidate_dates == exact_date
        ]

        if not date_filtered.empty:
            return date_filtered

    # -------------------------------------------------
    # 1. YEAR
    # -------------------------------------------------
    year_match = re.search(
        r"\b(18|19|20)\d{2}\b",
        reply
    )

    if year_match:
        year = year_match.group()

        candidate_years = pd.to_datetime(
            filtered["date"],
            errors="coerce"

In [214]:
# Restore intent routing sets

RESULTS_INTENTS = {
    "home_team_score",
    "away_team_score",
    "match_date",
    "match_location",
    "match_score",
    "match_winner",
    "neutral_status",
    "total_goals",
    "tournament",
    "head_to_head_matches",
    "team_match_history",
    "team_wins",
    "team_goals_scored",
    "team_goals_conceded"
}

GOALSCORER_INTENTS = {
    "scorer",
    "goal_minute",
    "penalty_status",
    "own_goal_status"
}

SHOOTOUT_INTENTS = {
    "shootout_winner",
    "first_shooter"
}

print("Intent routing sets restored.")

Intent routing sets restored.


In [215]:
# Final interactive football QA chatbot

question = input("Ask a football question: ")

football_qa(question)

Ask a football question: who won the world cup in 2018

Assistant: France won the FIFA World Cup final 4-2 against Croatia.


In [216]:
# Define intent groups for unified chatbot routing

RESULTS_INTENTS = {
    "away_team_score",
    "home_team_score",
    "match_date",
    "match_location",
    "match_score",
    "match_winner",
    "neutral_status",
    "total_goals",
    "tournament"
}

GOALSCORER_INTENTS = {
    "scorer",
    "goal_minute",
    "own_goal_status",
    "penalty_status"
}

SHOOTOUT_INTENTS = {
    "shootout_winner",
    "first_shooter"
}

print("Results intents:", len(RESULTS_INTENTS))
print("Goalscorer intents:", len(GOALSCORER_INTENTS))
print("Shootout intents:", len(SHOOTOUT_INTENTS))

print(
    "Total routed intents:",
    len(RESULTS_INTENTS)
    + len(GOALSCORER_INTENTS)
    + len(SHOOTOUT_INTENTS)
)

Results intents: 9
Goalscorer intents: 4
Shootout intents: 2
Total routed intents: 15


In [217]:
# Cell 74 - Test unified Random Forest chatbot

test_questions = [
    "Who won the 2022 World Cup final?",
    "At what minute did Christian Eriksen score against Slovenia?",
    "Who won the penalty shootout between Uruguay and Ghana?"
]

for question in test_questions:
    print("\n" + "=" * 70)
    print("Question:", question)

    football_qa(question)


Question: Who won the 2022 World Cup final?

Assistant: Argentina won the FIFA World Cup final on penalties after a 3-3 draw between Argentina and France.

Question: At what minute did Christian Eriksen score against Slovenia?

Assistant: Christian Eriksen scored in the 17th minute.

Question: Who won the penalty shootout between Uruguay and Ghana?
Possible shootout records: 1

Assistant: Uruguay won the penalty shootout between Uruguay and Ghana on 2010-07-02.


In [218]:
# Cell 75 - Unified chatbot test bank for all 15 intents

test_bank = [
    # Results intents
    "What was the away score when United States played Costa Rica on 2005-07-12?",
    "How many goals did the home team Togo score against Uganda?",
    "When did Poland play United States on 2002-06-14?",
    "Where was the Cameroon vs Saudi Arabia match played?",
    "What was the score between Turkey and Latvia on 2013-05-28?",
    "Who won the 2022 World Cup final?",
    "Was Turkey versus South Korea played on neutral ground?",
    "How many total goals were scored between Uzbekistan and Indonesia on 1997-06-20?",
    "Which tournament was the Bosnia and Herzegovina vs France match part of?",

    # Goalscorer intents
    "Which player scored for Paraguay against Chile?",
    "At what minute did Christian Eriksen score against Slovenia?",
    "Did Jordi Alba score an own goal against Italy?",
    "Did Georgios Samaras convert a penalty against Germany on 2012-06-22?",

    # Shootout intents
    "Who won the penalty shootout between Uruguay and Ghana?",
    "Who took the first penalty in the shootout between Uruguay and Ghana?"
]

for i, question in enumerate(test_bank, 1):
    print("\n" + "=" * 80)
    print(f"TEST {i}")
    print("Question:", question)

    football_qa(question)


TEST 1
Question: What was the away score when United States played Costa Rica on 2005-07-12?

Assistant: Costa Rica scored 0 goal(s).

TEST 2
Question: How many goals did the home team Togo score against Uganda?

Assistant: I found matches from multiple years. Do you remember the year? Options: 1965, 2000, 2014, 2015, 2016

You: 2000

Assistant: Togo scored 3 goal(s).

TEST 3
Question: When did Poland play United States on 2002-06-14?

Assistant: The match was played on 2002-06-14.

TEST 4
Question: Where was the Cameroon vs Saudi Arabia match played?

Assistant: I found matches from multiple years. Do you remember the year? Options: 1982, 1985, 1992, 2002

You: 1982

Assistant: The match was played in Riyadh, Saudi Arabia.

TEST 5
Question: What was the score between Turkey and Latvia on 2013-05-28?

Assistant: Turkey 3 - 3 Latvia.

TEST 6
Question: Who won the 2022 World Cup final?

Assistant: Argentina won the FIFA World Cup final on penalties after a 3-3 draw between Argentina and

In [219]:
question = "How many goals did the home team Togo score against Uganda?"

print("Detected roles:")
print(extract_team_role_constraints(question))

candidates = find_result_candidates(question)

print("\nMatches found:", len(candidates))

display(
    candidates[
        [
            "date",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
            "tournament"
        ]
    ]
)

Detected roles:
{'home_team': 'Togo', 'away_team': None}

Matches found: 5


,date,home_team,away_team,home_score,away_score,tournament
6357,1965-07-20,Togo,Uganda,1,1,All-African Games
24957,2000-10-08,Togo,Uganda,3,0,African Cup of Nations qualification
38230,2014-10-15,Togo,Uganda,1,0,African Cup of Nations qualification
39309,2015-11-12,Togo,Uganda,0,1,FIFA World Cup qualification
40095,2016-10-04,Togo,Uganda,1,0,Friendly


In [220]:
football_qa(
    "How many goals did the home team Togo score against Uganda?"
)


Assistant: I found matches from multiple years. Do you remember the year? Options: 1965, 2000, 2014, 2015, 2016

You: 2000

Assistant: Togo scored 3 goal(s).


In [221]:
# Cell 77 - Inspect exact-date Uzbekistan vs Indonesia records

question = (
    "How many total goals were scored between "
    "Uzbekistan and Indonesia on 1997-06-20?"
)

candidates = find_result_candidates(question)

print("Matches found:", len(candidates))

display(
    candidates[
        [
            "date",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
            "tournament",
            "city",
            "country",
            "neutral"
        ]
    ]
)

Matches found: 1


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
22096,1997-06-20,Uzbekistan,Indonesia,3,0,FIFA World Cup qualification,Tashkent,Uzbekistan,False


In [222]:
football_qa(
    "Who won between Argentina and Croatia?"
)


Assistant: I found matches from multiple years. Do you remember the year? Options: 1994, 1998, 2006, 2014, 2018, 2022

You: 2014

Assistant: Argentina won the match 2-1.


In [223]:
# Cell 81 - Stress-test major tournament final inference

major_final_tests = [
    "Who won the 2022 World Cup final?",
    "Who won the Euro 2020 final?",
    "Who won the Euro 2024 final?",
    "Who won the 2024 Copa America final?",
    "Who won the 2023 AFCON final?",
    "Who won the 2023 Asian Cup final?",
    "Who won the 2023 Gold Cup final?",
    "Who won the 2024 OFC Nations Cup final?",
    "Who won the 2025 UEFA Nations League final?",
    "Who won the 2025 CONCACAF Nations League final?"
]

for question in major_final_tests:
    print("\n" + "=" * 80)
    print("Question:", question)

    final_match, error = find_tournament_final(question)

    if error:
        print("Error:", error)
        continue

    print(
        "Detected final:",
        final_match["date"],
        "|",
        final_match["home_team"],
        final_match["home_score"],
        "-",
        final_match["away_score"],
        final_match["away_team"],
        "|",
        final_match["tournament"]
    )

    print(
        "Resolved answer:",
        resolve_final_winner(final_match)
    )


Question: Who won the 2022 World Cup final?
Detected final: 2022-12-18 | Argentina 3 - 3 France | FIFA World Cup
Resolved answer: Argentina won the FIFA World Cup final on penalties after a 3-3 draw between Argentina and France.

Question: Who won the Euro 2020 final?
Detected final: 2021-07-11 | England 1 - 1 Italy | UEFA Euro
Resolved answer: Italy won the UEFA Euro final on penalties after a 1-1 draw between England and Italy.

Question: Who won the Euro 2024 final?
Detected final: 2024-07-14 | Spain 2 - 1 England | UEFA Euro
Resolved answer: Spain won the UEFA Euro final 2-1 against England.

Question: Who won the 2024 Copa America final?
Detected final: 2024-07-14 | Argentina 1 - 0 Colombia | Copa América
Resolved answer: Argentina won the Copa América final 1-0 against Colombia.

Question: Who won the 2023 AFCON final?
Detected final: 2024-02-11 | Ivory Coast 2 - 1 Nigeria | African Cup of Nations
Resolved answer: Ivory Coast won the African Cup of Nations final 2-1 against Nige

In [224]:
# Cell 83 - Find major tournament final with edition-year handling

def find_tournament_final(question):
    question_normalized = normalize_text(question)

    tournament = detect_major_tournament(question)

    if tournament is None:
        return None, "I couldn't identify a supported major tournament."

    # Extract edition year mentioned by user
    year_match = re.search(r"\b(18|19|20)\d{2}\b", question)

    if not year_match:
        return None, "I couldn't identify the tournament year."

    edition_year = year_match.group()



    # Check whether this edition was actually played
    # in a different calendar year
    actual_year = TOURNAMENT_EDITION_YEAR_OVERRIDES.get(
        (tournament, edition_year),
        edition_year
    )

    edition_matches = results_df[
        (results_df["tournament"] == tournament)
        &
        (
            results_df["date"]
            .astype(str)
            .str.startswith(actual_year)
        )
    ].copy()

    if edition_matches.empty:
        return None, (
            f"I couldn't find {tournament} matches for the "
            f"{edition_year} edition."
        )

    edition_matches = edition_matches.sort_values("date")

    # Latest chronological match = inferred final
    final_match = edition_matches.iloc[-1]

    return final_match, None

In [225]:
test_questions = [
    "Who won the 2025 Nations League final?",
    "Who won the 2025 UEFA Nations League final?",
    "Who won the 2025 CONCACAF Nations League final?",
    "Who won the 2022 World Cup final?"
]

for question in test_questions:
    print(
        question,
        "->",
        detect_generic_nations_league(question)
    )

Who won the 2025 Nations League final? -> True
Who won the 2025 UEFA Nations League final? -> False
Who won the 2025 CONCACAF Nations League final? -> False
Who won the 2022 World Cup final? -> False


In [226]:
test_questions = [
    "Who won the 2025 Nations League final?",
    "Who won the Nations League match between Portugal and Spain?",
    "Who won the Nations League match between Mexico and Panama?"
]

for question in test_questions:
    print(
        question,
        "->",
        infer_nations_league_from_teams(question)
    )

Who won the 2025 Nations League final? -> None
Who won the Nations League match between Portugal and Spain? -> UEFA Nations League
Who won the Nations League match between Mexico and Panama? -> CONCACAF Nations League


In [227]:
football_qa(
    "Who won the 2025 Nations League final?"
)


Assistant: Do you mean the UEFA Nations League or the CONCACAF Nations League?

You: concacaf

Assistant: Mexico won the CONCACAF Nations League final 2-1 against Panama.


In [228]:
football_qa(
    "Who won the Nations League match between Portugal and Spain?"
)


Assistant: I identified this as the UEFA Nations League based on the teams mentioned.

Assistant: I found matches from multiple years. Do you remember the year? Options: 2022, 2025

You: 2022

Assistant: Do you remember which country the match was played in? Options: Portugal, Spain

You: spain

Assistant: The match ended in a 1-1 draw.


In [229]:
football_qa(
    "Who won the Nations League match between Mexico and Panama?"
)


Assistant: I identified this as the CONCACAF Nations League based on the teams mentioned.

Assistant: I found matches from multiple years. Do you remember the year? Options: 2019, 2023, 2024, 2025

You: 2023

Assistant: Mexico won the match 1-0.


In [230]:
football_qa(
    "Who took the first penalty in the shootout between Lesotho and Zambia?"
)

Possible shootout records: 1

Assistant: The first shooter is not recorded for the Lesotho vs Zambia shootout on 2000-06-11.


In [231]:
# Cell 88 - Inspect historical/former team name mappings

print("Former names dataset shape:", former_names_df.shape)
print("\nColumns:", former_names_df.columns.tolist())

display(former_names_df.head(20))

Former names dataset shape: (36, 4)

Columns: ['current', 'former', 'start_date', 'end_date']


,current,former,start_date,end_date
0,Benin,Dahomey,1959-11-08,1975-11-30
1,Burkina Faso,Upper Volta,1960-04-14,1984-08-04
2,Curaçao,Netherlands Antilles,1957-03-03,2010-10-10
3,Czechoslovakia,Bohemia,1903-04-05,1919-01-01
4,Czechoslovakia,Bohemia and Moravia,1939-01-01,1945-05-01
5,Czechoslovakia,Representation of Czechs and Slovaks,1993-03-24,1993-11-17
6,DR Congo,Belgian Congo,1948-05-25,1956-01-02
7,DR Congo,Congo-Léopoldville,1963-04-12,1964-07-19
8,DR Congo,Congo-Kinshasa,1965-01-09,1970-11-24
9,DR Congo,Zaïre,1971-01-10,1997-04-27


In [232]:
# Check date types and missing values

print("\nData types:")
print(former_names_df.dtypes)

print("\nMissing values:")
print(former_names_df.isna().sum())


Data types:
current       object
former        object
start_date    object
end_date      object
dtype: object

Missing values:
current       0
former        0
start_date    0
end_date      0
dtype: int64


In [233]:
# Cell 91 - Test former-name detection

historical_name_tests = [
    "Who won between Zaire and Zambia?",
    "What was the score between Upper Volta and Ghana?",
    "When did Dahomey play Nigeria?",
    "Did Swaziland play South Africa?",
    "Who won between DR Congo and Angola?"
]

for question in historical_name_tests:
    print("\nQuestion:", question)
    print(
        "Detected:",
        detect_former_team_names(question)
    )


Question: Who won between Zaire and Zambia?
Detected: [{'former': 'Zaïre', 'current': 'DR Congo', 'start_date': Timestamp('1971-01-10 00:00:00'), 'end_date': Timestamp('1997-04-27 00:00:00')}]

Question: What was the score between Upper Volta and Ghana?
Detected: [{'former': 'Upper Volta', 'current': 'Burkina Faso', 'start_date': Timestamp('1960-04-14 00:00:00'), 'end_date': Timestamp('1984-08-04 00:00:00')}]

Question: When did Dahomey play Nigeria?
Detected: [{'former': 'Dahomey', 'current': 'Benin', 'start_date': Timestamp('1959-11-08 00:00:00'), 'end_date': Timestamp('1975-11-30 00:00:00')}]

Question: Did Swaziland play South Africa?
Detected: [{'former': 'Swaziland', 'current': 'Eswatini', 'start_date': Timestamp('1968-05-01 00:00:00'), 'end_date': Timestamp('2018-04-19 00:00:00')}]

Question: Who won between DR Congo and Angola?
Detected: []


In [234]:
# Cell 93 - Test historical-name resolution

historical_tests = [
    "Who won between Zaire and Zambia?",
    "What was the score between Upper Volta and Ghana?",
    "When did Dahomey play Nigeria?",
    "Did Swaziland play South Africa?",
    "Who won between DR Congo and Angola?"
]

for question in historical_tests:

    resolved, constraints = resolve_former_team_names(question)

    print("\nOriginal :", question)
    print("Resolved :", resolved)
    print("Constraints:", constraints)


Original : Who won between Zaire and Zambia?
Resolved : Who won between DR Congo and Zambia?
Constraints: [{'former': 'Zaïre', 'current': 'DR Congo', 'start_date': Timestamp('1971-01-10 00:00:00'), 'end_date': Timestamp('1997-04-27 00:00:00')}]

Original : What was the score between Upper Volta and Ghana?
Resolved : What was the score between Burkina Faso and Ghana?
Constraints: [{'former': 'Upper Volta', 'current': 'Burkina Faso', 'start_date': Timestamp('1960-04-14 00:00:00'), 'end_date': Timestamp('1984-08-04 00:00:00')}]

Original : When did Dahomey play Nigeria?
Resolved : When did Benin play Nigeria?
Constraints: [{'former': 'Dahomey', 'current': 'Benin', 'start_date': Timestamp('1959-11-08 00:00:00'), 'end_date': Timestamp('1975-11-30 00:00:00')}]

Original : Did Swaziland play South Africa?
Resolved : Did Eswatini play South Africa?
Constraints: [{'former': 'Swaziland', 'current': 'Eswatini', 'start_date': Timestamp('1968-05-01 00:00:00'), 'end_date': Timestamp('2018-04-19 00:

In [235]:
historical_retrieval_tests = [
    "Who won between Zaire and Zambia?",
    "What was the score between Upper Volta and Ghana?",
    "When did Dahomey play Nigeria?",
    "Who won between DR Congo and Angola?"
]

for question in historical_retrieval_tests:

    candidates = find_result_candidates(question)

    print("\n" + "=" * 70)
    print("Question:", question)
    print("Matches found:", len(candidates))

    if not candidates.empty:
        print(
            "Date range:",
            candidates["date"].min(),
            "to",
            candidates["date"].max()
        )

        display(
            candidates[
                [
                    "date",
                    "home_team",
                    "away_team",
                    "home_score",
                    "away_score",
                    "tournament"
                ]
            ].head(10)
        )


Question: Who won between Zaire and Zambia?
Matches found: 17
Date range: 1971-10-13 to 1997-04-09


,date,home_team,away_team,home_score,away_score,tournament
8590,1971-10-13,Zambia,DR Congo,2,1,African Cup of Nations qualification
8604,1971-10-27,DR Congo,Zambia,3,0,African Cup of Nations qualification
9549,1973-11-04,Zambia,DR Congo,0,2,FIFA World Cup qualification
9572,1973-11-18,DR Congo,Zambia,2,1,FIFA World Cup qualification
9672,1974-03-12,DR Congo,Zambia,2,2,African Cup of Nations
9675,1974-03-14,DR Congo,Zambia,2,0,African Cup of Nations
11815,1979-06-24,Zambia,DR Congo,4,1,Friendly
11827,1979-06-30,DR Congo,Zambia,1,2,Friendly
12829,1981-06-26,Zambia,DR Congo,3,1,Friendly
12834,1981-06-28,Zambia,DR Congo,2,2,Friendly



Question: What was the score between Upper Volta and Ghana?
Matches found: 5
Date range: 1967-10-01 to 1984-08-03


,date,home_team,away_team,home_score,away_score,tournament
7152,1967-10-01,Ghana,Burkina Faso,5,2,Friendly
10056,1975-03-23,Ghana,Burkina Faso,3,1,Friendly
11297,1978-03-10,Ghana,Burkina Faso,3,0,African Cup of Nations
13150,1982-02-19,Ghana,Burkina Faso,1,0,West African Cup
14229,1984-08-03,Burkina Faso,Ghana,0,2,Friendly



Question: When did Dahomey play Nigeria?
Matches found: 7
Date range: 1959-11-08 to 1972-12-20


,date,home_team,away_team,home_score,away_score,tournament
4947,1959-11-08,Benin,Nigeria,0,1,Friendly
4954,1959-11-28,Nigeria,Benin,10,1,Friendly
5696,1963-01-27,Benin,Nigeria,1,1,Friendly
5699,1963-02-02,Nigeria,Benin,4,1,Friendly
6538,1965-12-11,Nigeria,Benin,0,1,Friendly
6555,1966-02-27,Benin,Nigeria,1,2,Friendly
9131,1972-12-20,Nigeria,Benin,3,0,Friendly



Question: Who won between DR Congo and Angola?
Matches found: 15
Date range: 1987-03-29 to 2024-01-06


,date,home_team,away_team,home_score,away_score,tournament
15638,1987-03-29,DR Congo,Angola,3,0,African Cup of Nations qualification
15657,1987-04-11,Angola,DR Congo,1,0,African Cup of Nations qualification
16494,1988-12-30,DR Congo,Angola,0,0,Friendly
27454,2003-08-20,Angola,DR Congo,0,2,Friendly
28167,2004-05-23,DR Congo,Angola,1,3,Friendly
29760,2006-01-25,Angola,DR Congo,0,0,African Cup of Nations
31139,2007-08-22,DR Congo,Angola,3,1,Friendly
35068,2011-08-27,Angola,DR Congo,1,2,Friendly
39275,2015-11-07,Angola,DR Congo,1,0,Friendly
39561,2016-03-26,DR Congo,Angola,2,1,African Cup of Nations qualification


In [236]:
football_qa(
    "At what minutes did Christian Eriksen score against Republic of Ireland on 2017-11-14?"
)


Assistant: I couldn't find a match that fits the information in your question.


In [237]:
# 1. Exact-date retrieval
football_qa(
    "How many total goals were scored between Uzbekistan and Indonesia on 1997-06-20?"
)


Assistant: A total of 3 goal(s) were scored.


In [238]:
# 2. Explicit home-team role
football_qa(
    "How many goals did the home team Togo score against Uganda?"
)


Assistant: I found matches from multiple years. Do you remember the year? Options: 1965, 2000, 2014, 2015, 2016

You: 2015

Assistant: Togo scored 0 goal(s).


In [239]:
# 3. Multiple goals by same scorer
football_qa(
    "At what minutes did Christian Eriksen score against Republic of Ireland on 2017-11-14?"
)


Assistant: I couldn't find a match that fits the information in your question.


In [240]:
print("Test 1:", extract_exact_date("1974-03-12"))
print("Test 2:", extract_exact_date(" 1974-03-12 "))
print("Test 3:", extract_exact_date("The date was 1974-03-12"))

Test 1: 1974-03-12
Test 2: 1974-03-12
Test 3: 1974-03-12


In [241]:
question = "Who won between Zaire and Zambia?"

candidates = find_result_candidates(question)

# Simulate first clarification
candidates_1974 = apply_smart_clarification(
    candidates,
    "1974"
)

print("After 1974:", len(candidates_1974))

# Simulate exact-date clarification
candidates_exact = apply_smart_clarification(
    candidates_1974,
    "1974-03-12"
)

print("After exact date:", len(candidates_exact))

display(
    candidates_exact[
        [
            "date",
            "home_team",
            "away_team",
            "home_score",
            "away_score"
        ]
    ]
)

After 1974: 2
After exact date: 1


,date,home_team,away_team,home_score,away_score
9672,1974-03-12,DR Congo,Zambia,2,2


In [242]:
# 4. Historical team name
football_qa(
    "Who won between Zaire and Zambia?"
)


Assistant: I found matches across 10 different years. Do you remember roughly which year it was?

You: 1974

Assistant: I still found more than one possible match. Do you remember the exact date? Options: 1974-03-12, 1974-03-14

You: 1974-03-12

Assistant: The match ended in a 2-2 draw.


In [243]:
# Debug exact-date clarification

question = "Who won between Zaire and Zambia?"

candidates = find_result_candidates(question)

candidates_1974 = apply_smart_clarification(
    candidates,
    "1974"
)

reply = "1974-03-12"

print("1. Raw reply:")
print(repr(reply))

print("\n2. extract_exact_date result:")
exact_date = extract_exact_date(reply)
print(repr(exact_date))
print("Type:", type(exact_date))

print("\n3. Candidate dates:")
candidate_dates = pd.to_datetime(
    candidates_1974["date"],
    errors="coerce"
).dt.strftime("%Y-%m-%d")

print(candidate_dates.tolist())

print("\n4. Exact comparison:")
print((candidate_dates == exact_date).tolist())

print("\n5. Rows matching exact date:")
date_filtered = candidates_1974[
    candidate_dates == exact_date
]

print("Count:", len(date_filtered))

display(
    date_filtered[
        [
            "date",
            "home_team",
            "away_team",
            "home_score",
            "away_score"
        ]
    ]
)

1. Raw reply:
'1974-03-12'

2. extract_exact_date result:
'1974-03-12'
Type: <class 'str'>

3. Candidate dates:
['1974-03-12', '1974-03-14']

4. Exact comparison:
[True, False]

5. Rows matching exact date:
Count: 1


,date,home_team,away_team,home_score,away_score
9672,1974-03-12,DR Congo,Zambia,2,2


In [244]:
question = "Who won between Zaire and Zambia?"

candidates = find_result_candidates(question)

candidates_1974 = apply_smart_clarification(
    candidates,
    "1974"
)

candidates_exact = apply_smart_clarification(
    candidates_1974,
    "1974-03-12"
)

print("After 1974:", len(candidates_1974))
print("After exact date:", len(candidates_exact))

display(
    candidates_exact[
        [
            "date",
            "home_team",
            "away_team",
            "home_score",
            "away_score"
        ]
    ]
)

After 1974: 2
After exact date: 1


,date,home_team,away_team,home_score,away_score
9672,1974-03-12,DR Congo,Zambia,2,2


In [245]:
print(inspect.getsource(apply_smart_clarification))

def apply_smart_clarification(candidates, reply):
    reply_normalized = normalize_text(reply)
    filtered = candidates.copy()

    # -------------------------------------------------
    # 0. EXACT DATE - MUST COME BEFORE YEAR
    # -------------------------------------------------
    exact_date = extract_exact_date(reply)

    if exact_date:
        candidate_dates = pd.to_datetime(
            filtered["date"],
            errors="coerce"
        ).dt.strftime("%Y-%m-%d")

        date_filtered = filtered[
            candidate_dates == exact_date
        ]

        if not date_filtered.empty:
            return date_filtered

    # -------------------------------------------------
    # 1. YEAR
    # -------------------------------------------------
    year_match = re.search(
        r"\b(18|19|20)\d{2}\b",
        reply
    )

    if year_match:
        year = year_match.group()

        candidate_years = pd.to_datetime(
            filtered["date"],
            errors="coerce"

In [246]:
question = "Who won between Zaire and Zambia?"

candidates = find_result_candidates(question)

candidates_1974 = apply_smart_clarification(
    candidates,
    "1974"
)

candidates_exact = apply_smart_clarification(
    candidates_1974,
    "1974-03-12"
)

print("After 1974:", len(candidates_1974))
print("After exact date:", len(candidates_exact))

After 1974: 2
After exact date: 1


In [247]:
# Test 5 - World Cup final + shootout
football_qa(
    "Who won the 2022 World Cup final?"
)


Assistant: Argentina won the FIFA World Cup final on penalties after a 3-3 draw between Argentina and France.


In [248]:
# Test 6 - Tournament edition played in following year
football_qa(
    "Who won the 2023 AFCON final?"
)


Assistant: Ivory Coast won the African Cup of Nations final 2-1 against Nigeria.


In [249]:
# Test 7 - Ambiguous Nations League
football_qa(
    "Who won the 2025 Nations League final?"
)


Assistant: Do you mean the UEFA Nations League or the CONCACAF Nations League?

You: uefa

Assistant: Portugal won the UEFA Nations League final on penalties after a 2-2 draw between Portugal and Spain.


In [250]:
# Test 8 - UEFA Nations League from teams
football_qa(
    "Who won the Nations League match between Portugal and Spain?"
)


Assistant: I identified this as the UEFA Nations League based on the teams mentioned.

Assistant: I found matches from multiple years. Do you remember the year? Options: 2022, 2025

You: 2022

Assistant: Do you remember which country the match was played in? Options: Portugal, Spain

You: spain

Assistant: The match ended in a 1-1 draw.


In [251]:
# Test 9 - Missing first-shooter information
football_qa(
    "Who took the first penalty in the shootout between Lesotho and Zambia?"
)

Possible shootout records: 1

Assistant: The first shooter is not recorded for the Lesotho vs Zambia shootout on 2000-06-11.


In [252]:
# Test 10 - Own goal
football_qa(
    "Did Jordi Alba score an own goal against Italy?"
)


Assistant: Jordi Alba's goal was not an own goal.


In [253]:
# Test 11 - Penalty
football_qa(
    "Did Georgios Samaras convert a penalty against Germany on 2012-06-22?"
)


Assistant: Georgios Samaras's goal was not a penalty.


In [254]:
# Test 12 - Normal score retrieval
football_qa(
    "What was the score between Turkey and Latvia on 2013-05-28?"
)


Assistant: Turkey 3 - 3 Latvia.


In [255]:
football_qa(
    "What was the score between Turkey and Latvia on 2013-05-28?"
)


Assistant: Turkey 3 - 3 Latvia.


In [256]:
football_qa(
    "At what minutes did Christian Eriksen score against Republic of Ireland on 2017-11-14?"
)


Assistant: I couldn't find a match that fits the information in your question.


In [257]:
football_qa(
    "Who won the penalty shootout between Uruguay and Ghana?"
)

Possible shootout records: 1

Assistant: Uruguay won the penalty shootout between Uruguay and Ghana on 2010-07-02.


In [258]:
# Improved recent-form / last-N-match detector

def detect_recent_form_question(question):
    q = normalize_text(question)

    patterns = [
        # last 5 matches / last 5 international matches / last 10 games
        r"\blast\s+\d+\s+(?:international\s+)?(?:matches|games)\b",

        # last five matches / last five international games
        r"\blast\s+(?:one|two|three|four|five|six|seven|eight|nine|ten)\s+"
        r"(?:international\s+)?(?:matches|games)\b",

        # recent matches / recent international matches
        r"\brecent\s+(?:international\s+)?(?:matches|games)\b",

        # recent form / current form
        r"\b(?:recent|current)\s+form\b",

        # form in the last 5 games
        r"\bform\s+(?:in|over)\s+the\s+last\b"
    ]

    return any(
        re.search(pattern, q)
        for pattern in patterns
    )

In [259]:
test_questions = [
    "What are Argentina's last 5 international matches?",
    "What are Argentina's last 5 matches?",
    "Show me Argentina's recent international games",
    "How is Argentina's recent form?",
    "Who won between Argentina and Brazil?"
]

for q in test_questions:
    print(q, "->", detect_recent_form_question(q))

What are Argentina's last 5 international matches? -> True
What are Argentina's last 5 matches? -> True
Show me Argentina's recent international games -> True
How is Argentina's recent form? -> True
Who won between Argentina and Brazil? -> False


In [260]:
# Extract team and requested number of recent matches

def extract_recent_form_request(question):
    entities = extract_basic_entities(question)

    teams = entities["teams"]
    team = teams[0] if teams else None

    # Default = 5
    match_count = 5

    number_match = re.search(
        r"\blast\s+(\d+)\b",
        normalize_text(question)
    )

    if number_match:
        match_count = int(number_match.group(1))

    # Keep it reasonable
    match_count = max(1, min(match_count, 10))

    return team, match_count

In [261]:
# Retrieve recent matches and calculate form statistics

def get_recent_team_form(team, match_count=5):
    team_matches = results_df[
        (results_df["home_team"] == team)
        |
        (results_df["away_team"] == team)
    ].copy()

    if team_matches.empty:
        return None

    team_matches["date_dt"] = pd.to_datetime(
        team_matches["date"],
        errors="coerce"
    )

    team_matches = (
        team_matches
        .sort_values("date_dt", ascending=False)
        .head(match_count)
        .sort_values("date_dt")
    )

    form = []
    wins = 0
    draws = 0
    losses = 0
    goals_scored = 0
    goals_conceded = 0

    match_lines = []

    for _, row in team_matches.iterrows():
        home = row["home_team"]
        away = row["away_team"]
        home_score = int(row["home_score"])
        away_score = int(row["away_score"])

        if team == home:
            team_score = home_score
            opponent_score = away_score
        else:
            team_score = away_score
            opponent_score = home_score

        goals_scored += team_score
        goals_conceded += opponent_score

        if team_score > opponent_score:
            result = "W"
            wins += 1
        elif team_score < opponent_score:
            result = "L"
            losses += 1
        else:
            result = "D"
            draws += 1

        form.append(result)

        match_lines.append(
            f"{row['date']}: "
            f"{home} {home_score}-{away_score} {away}"
        )

    return {
        "team": team,
        "matches": match_lines,
        "form": form,
        "wins": wins,
        "draws": draws,
        "losses": losses,
        "goals_scored": goals_scored,
        "goals_conceded": goals_conceded
    }

In [262]:
# Generate recent-form answer

def generate_recent_form_answer(question):
    team, match_count = extract_recent_form_request(question)

    if team is None:
        return (
            "I couldn't identify which team you want "
            "recent form for."
        )

    stats = get_recent_team_form(
        team,
        match_count
    )

    if stats is None:
        return (
            f"I couldn't find recent match data for {team}."
        )

    matches_text = "\n".join(
        f"{i+1}. {match}"
        for i, match in enumerate(stats["matches"])
    )

    form_text = "-".join(stats["form"])

    return (
        f"{team}'s last {len(stats['matches'])} international matches:\n\n"
        f"{matches_text}\n\n"
        f"Form: {form_text}\n"
        f"Record: {stats['wins']} wins, "
        f"{stats['draws']} draws, "
        f"{stats['losses']} losses\n"
        f"Goals: {stats['goals_scored']} scored, "
        f"{stats['goals_conceded']} conceded."
    )

In [263]:
# FINAL FOOTBALL QA ROUTER
# Keep this as the last football_qa definition

def football_qa(question):

    # -------------------------------------------------
    # 1. Recent form / last-N matches
    # -------------------------------------------------
    if detect_recent_form_question(question):
        print(
            "\nAssistant:",
            generate_recent_form_answer(question)
        )
        return

    # -------------------------------------------------
    # 2. Nations League ambiguity
    # -------------------------------------------------
    question = resolve_nations_league_reference(question)

    # -------------------------------------------------
    # 3. Random Forest intent prediction
    # -------------------------------------------------
    intent = predict_intent(question)

    if DEBUG:
        print(f"\nPredicted intent: {intent}")

    question_normalized = normalize_text(question)
    tournament = detect_major_tournament(question)

    # -------------------------------------------------
    # 4. Major tournament winner/final questions
    # -------------------------------------------------
    if (
        intent == "match_winner"
        and tournament is not None
        and (
            "final" in question_normalized
            or is_tournament_winner_question(question)
        )
    ):
        final_match, error = find_tournament_final(question)

        if error is None:
            print(
                "\nAssistant:",
                resolve_final_winner(final_match)
            )
            return

    # -------------------------------------------------
    # 5. Results
    # -------------------------------------------------
    if intent in RESULTS_INTENTS:
        ask_results_question(question)
        return

    # -------------------------------------------------
    # 6. Goalscorers
    # -------------------------------------------------
    if intent in GOALSCORER_INTENTS:
        ask_goalscorer_question(question)
        return

    # -------------------------------------------------
    # 7. Shootouts
    # -------------------------------------------------
    if intent in SHOOTOUT_INTENTS:
        ask_shootout_question(question)
        return

    print(
        "\nAssistant: I understood the question, "
        "but I don't currently have an answer handler "
        "for this type of question."
    )

In [264]:
football_qa(
    "What are Argentina's last 5 international matches?"
)


Assistant: Argentina's last 5 international matches:

1. 2026-07-03: Argentina 3-2 Cape Verde
2. 2026-07-07: Argentina 3-2 Egypt
3. 2026-07-11: Argentina 3-1 Switzerland
4. 2026-07-15: England 1-2 Argentina
5. 2026-07-19: Spain 1-0 Argentina

Form: W-W-W-W-L
Record: 4 wins, 0 draws, 1 losses
Goals: 11 scored, 7 conceded.


In [265]:
football_qa(
    "How is Argentina's recent form?"
)


Assistant: Argentina's last 5 international matches:

1. 2026-07-03: Argentina 3-2 Cape Verde
2. 2026-07-07: Argentina 3-2 Egypt
3. 2026-07-11: Argentina 3-1 Switzerland
4. 2026-07-15: England 1-2 Argentina
5. 2026-07-19: Spain 1-0 Argentina

Form: W-W-W-W-L
Record: 4 wins, 0 draws, 1 losses
Goals: 11 scored, 7 conceded.


In [266]:
question = input("Ask a football question: ")
football_qa(question)

Ask a football question: who won the copa america 2021

Assistant: Argentina won the Copa América final 1-0 against Brazil.


In [267]:
def get_shootout_for_match(row):
    date = str(row["date"])
    home_team = row["home_team"]
    away_team = row["away_team"]

    shootout = shootouts_df[
        (shootouts_df["date"].astype(str) == date)
        &
        (
            (
                (shootouts_df["home_team"] == home_team)
                &
                (shootouts_df["away_team"] == away_team)
            )
            |
            (
                (shootouts_df["home_team"] == away_team)
                &
                (shootouts_df["away_team"] == home_team)
            )
        )
    ]

    if shootout.empty:
        return None

    return shootout.iloc[0]

In [268]:
# Find a shootout corresponding to a results match

def get_shootout_for_match(row):
    date = str(row["date"])
    home_team = row["home_team"]
    away_team = row["away_team"]

    shootout = shootouts_df[
        (shootouts_df["date"].astype(str) == date)
        &
        (
            (
                (shootouts_df["home_team"] == home_team)
                &
                (shootouts_df["away_team"] == away_team)
            )
            |
            (
                (shootouts_df["home_team"] == away_team)
                &
                (shootouts_df["away_team"] == home_team)
            )
        )
    ]

    if shootout.empty:
        return None

    return shootout.iloc[0]

In [269]:
# Find a shootout corresponding to a results match

def get_shootout_for_match(row):
    date = str(row["date"])
    home_team = row["home_team"]
    away_team = row["away_team"]

    shootout = shootouts_df[
        (shootouts_df["date"].astype(str) == date)
        &
        (
            (
                (shootouts_df["home_team"] == home_team)
                &
                (shootouts_df["away_team"] == away_team)
            )
            |
            (
                (shootouts_df["home_team"] == away_team)
                &
                (shootouts_df["away_team"] == home_team)
            )
        )
    ]

    if shootout.empty:
        return None

    return shootout.iloc[0]

In [270]:
# Final result answer generator
# Adds shootout-aware handling for tied scores/winners

def generate_result_answer(intent, row):
    home_team = row["home_team"]
    away_team = row["away_team"]
    home_score = row["home_score"]
    away_score = row["away_score"]
    date = row["date"]
    tournament = row["tournament"]
    city = row["city"]
    country = row["country"]
    neutral = row["neutral"]

    if intent == "home_team_score":
        return f"{home_team} scored {home_score} goal(s)."

    elif intent == "away_team_score":
        return f"{away_team} scored {away_score} goal(s)."

    elif intent == "match_score":
        base_answer = (
            f"{home_team} {home_score} - {away_score} {away_team}."
        )

        if home_score == away_score:
            shootout = get_shootout_for_match(row)

            if shootout is not None:
                shootout_winner = shootout["winner"]

                return (
                    f"{base_answer} "
                    f"{shootout_winner} won the penalty shootout."
                )

        return base_answer

    elif intent == "match_winner":
        if home_score > away_score:
            return (
                f"{home_team} won the match "
                f"{home_score}-{away_score}."
            )

        elif away_score > home_score:
            return (
                f"{away_team} won the match "
                f"{away_score}-{home_score}."
            )

        else:
            shootout = get_shootout_for_match(row)

            if shootout is not None:
                shootout_winner = shootout["winner"]

                return (
                    f"The match ended {home_score}-{away_score}, "
                    f"and {shootout_winner} won the penalty shootout."
                )

            return (
                f"The match ended in a "
                f"{home_score}-{away_score} draw."
            )

    elif intent == "total_goals":
        total = home_score + away_score
        return f"A total of {total} goal(s) were scored."

    elif intent == "match_date":
        return f"The match was played on {date}."

    elif intent == "match_location":
        return f"The match was played in {city}, {country}."

    elif intent == "tournament":
        return f"The match was part of the {tournament}."

    elif intent == "neutral_status":
        neutral_text = "Yes" if bool(neutral) else "No"

        return (
            f"{neutral_text}, the match "
            f"{'was' if bool(neutral) else 'was not'} "
            f"played at a neutral venue."
        )

    return "Answer generation for this intent is not implemented yet."

In [271]:
print(
    "detect_recent_form_question exists:",
    "detect_recent_form_question" in globals()
)

print(
    "generate_recent_form_answer exists:",
    "generate_recent_form_answer" in globals()
)

print(
    "football_qa exists:",
    "football_qa" in globals()
)

if "detect_recent_form_question" in globals():
    print(
        "\nDetector test:",
        detect_recent_form_question(
            "argentina last 5 match results"
        )
    )

detect_recent_form_question exists: True
generate_recent_form_answer exists: True
football_qa exists: True

Detector test: False


In [272]:
# FINAL recent-form detector

def detect_recent_form_question(question):
    q = normalize_text(question)

    patterns = [
        # last 5 matches / last 5 games / last 5 match results / last 5 results
        r"\blast\s+\d+\s+(?:international\s+)?"
        r"(?:matches|games|match\s+results|game\s+results|results)\b",

        # last five matches / games / match results
        r"\blast\s+(?:one|two|three|four|five|six|seven|eight|nine|ten)\s+"
        r"(?:international\s+)?"
        r"(?:matches|games|match\s+results|game\s+results|results)\b",

        # recent matches / games / results
        r"\brecent\s+(?:international\s+)?"
        r"(?:matches|games|match\s+results|game\s+results|results)\b",

        # recent form / current form
        r"\b(?:recent|current)\s+form\b",

        # form in the last / form over the last
        r"\bform\s+(?:in|over)\s+the\s+last\b"
    ]

    return any(
        re.search(pattern, q)
        for pattern in patterns
    )

In [273]:
tests = [
    "argentina last 5 match results",
    "Argentina's last 5 international matches",
    "Argentina last 5 games",
    "show Argentina recent results",
    "how is Argentina's recent form?",
    "who won Argentina vs Brazil?"
]

for q in tests:
    print(q, "->", detect_recent_form_question(q))

argentina last 5 match results -> True
Argentina's last 5 international matches -> True
Argentina last 5 games -> True
show Argentina recent results -> True
how is Argentina's recent form? -> True
who won Argentina vs Brazil? -> False


In [274]:
question = input("Ask a football question: ")
football_qa(question)

Ask a football question: who won the worldcup in 2006

Assistant: Italy won the FIFA World Cup final on penalties after a 1-1 draw between Italy and France.


In [275]:
# FINAL KNOCKOUT-STAGE CONFIGURATION
# Added at the end of the notebook

def get_knockout_structure(tournament, edition_year):
    year = int(edition_year)

    # -------------------------------------------------
    # FIFA WORLD CUP
    # -------------------------------------------------
    if tournament == "FIFA World Cup":

        # 2026 onward: 48 teams, Round of 32 added
        if year >= 2026:
            return [
                ("Round of 32", 16),
                ("Round of 16", 8),
                ("Quarterfinal", 4),
                ("Semifinal", 2),
                ("Third-place playoff", 1),
                ("Final", 1)
            ]

        # Modern pre-2026 format
        return [
            ("Round of 16", 8),
            ("Quarterfinal", 4),
            ("Semifinal", 2),
            ("Third-place playoff", 1),
            ("Final", 1)
        ]

    # -------------------------------------------------
    # UEFA EURO
    # Modern 24-team format
    # No third-place playoff
    # -------------------------------------------------
    if tournament == "UEFA Euro" and year >= 2016:
        return [
            ("Round of 16", 8),
            ("Quarterfinal", 4),
            ("Semifinal", 2),
            ("Final", 1)
        ]

    # -------------------------------------------------
    # AFRICA CUP OF NATIONS
    # Modern 24-team format
    # -------------------------------------------------
    if tournament == "African Cup of Nations" and year >= 2019:
        return [
            ("Round of 16", 8),
            ("Quarterfinal", 4),
            ("Semifinal", 2),
            ("Third-place playoff", 1),
            ("Final", 1)
        ]

    # -------------------------------------------------
    # AFC ASIAN CUP
    # Modern 24-team format
    # -------------------------------------------------
    if tournament == "AFC Asian Cup" and year >= 2019:
        return [
            ("Round of 16", 8),
            ("Quarterfinal", 4),
            ("Semifinal", 2),
            ("Final", 1)
        ]

    # -------------------------------------------------
    # COPA AMERICA
    # Modern format starts knockout phase at QF
    # -------------------------------------------------
    if tournament == "Copa América":
        return [
            ("Quarterfinal", 4),
            ("Semifinal", 2),
            ("Third-place playoff", 1),
            ("Final", 1)
        ]

    # -------------------------------------------------
    # UEFA NATIONS LEAGUE
    # Finals tournament
    # -------------------------------------------------
    if tournament == "UEFA Nations League":
        return [
            ("Semifinal", 2),
            ("Third-place playoff", 1),
            ("Final", 1)
        ]

    return None

In [276]:
def get_tournament_edition_matches(tournament, edition_year):
    edition_year = str(edition_year)

    actual_year = TOURNAMENT_EDITION_YEAR_OVERRIDES.get(
        (tournament, edition_year),
        edition_year
    )

    matches = results_df[
        (results_df["tournament"] == tournament)
        &
        (
            results_df["date"]
            .astype(str)
            .str.startswith(actual_year)
        )
    ].copy()

    if matches.empty:
        return matches

    matches["date_dt"] = pd.to_datetime(
        matches["date"],
        errors="coerce"
    )

    return matches.sort_values("date_dt")

In [277]:
def infer_knockout_stage(row):
    tournament = row["tournament"]

    # Find edition year from match date
    actual_year = str(row["date"])[:4]

    # Reverse-map postponed editions where needed
    edition_year = actual_year

    for (mapped_tournament, official_year), played_year in (
        TOURNAMENT_EDITION_YEAR_OVERRIDES.items()
    ):
        if (
            mapped_tournament == tournament
            and played_year == actual_year
        ):
            edition_year = official_year
            break

    structure = get_knockout_structure(
        tournament,
        edition_year
    )

    if structure is None:
        return None

    edition_matches = get_tournament_edition_matches(
        tournament,
        edition_year
    )

    if edition_matches.empty:
        return None

    # Number of knockout matches expected
    knockout_count = sum(
        count for _, count in structure
    )

    if len(edition_matches) < knockout_count:
        return None

    # The last N matches are the knockout phase
    knockout_matches = edition_matches.tail(
        knockout_count
    ).copy()

    # Build stage labels in chronological order
    stage_labels = []

    for stage_name, count in structure:
        stage_labels.extend(
            [stage_name] * count
        )

    knockout_matches["inferred_stage"] = stage_labels

    # Identify the selected match
    selected = knockout_matches[
        (
            knockout_matches["date"].astype(str)
            == str(row["date"])
        )
        &
        (
            knockout_matches["home_team"]
            == row["home_team"]
        )
        &
        (
            knockout_matches["away_team"]
            == row["away_team"]
        )
    ]

    if selected.empty:
        return None

    return selected.iloc[0]["inferred_stage"]

In [278]:
stage_tests = [
    ("2021-06-28", "France", "Switzerland"),
    ("2022-12-13", "Argentina", "Croatia"),
    ("2022-12-17", "Croatia", "Morocco"),
    ("2022-12-18", "Argentina", "France"),
]

for date, home, away in stage_tests:

    test_rows = results_df[
        (results_df["date"].astype(str) == date)
        &
        (results_df["home_team"] == home)
        &
        (results_df["away_team"] == away)
    ]

    if test_rows.empty:
        print(date, home, "vs", away, "-> NOT FOUND")
        continue

    row = test_rows.iloc[0]

    print(
        date,
        home,
        "vs",
        away,
        "->",
        infer_knockout_stage(row)
    )

2021-06-28 France vs Switzerland -> Round of 16
2022-12-13 Argentina vs Croatia -> Semifinal
2022-12-17 Croatia vs Morocco -> Third-place playoff
2022-12-18 Argentina vs France -> Final


In [279]:
# FINAL result answer generator
# Includes:
# - shootout-aware results
# - knockout-stage context

def generate_result_answer(intent, row):
    home_team = row["home_team"]
    away_team = row["away_team"]
    home_score = row["home_score"]
    away_score = row["away_score"]
    date = row["date"]
    tournament = row["tournament"]
    city = row["city"]
    country = row["country"]
    neutral = row["neutral"]

    stage = infer_knockout_stage(row)

    stage_text = ""

    if stage:
        stage_text = (
            f" This was a {stage} match "
            f"at the {tournament}."
        )

    # -------------------------------------------------
    # HOME SCORE
    # -------------------------------------------------
    if intent == "home_team_score":
        return (
            f"{home_team} scored {home_score} goal(s)."
            f"{stage_text}"
        )

    # -------------------------------------------------
    # AWAY SCORE
    # -------------------------------------------------
    elif intent == "away_team_score":
        return (
            f"{away_team} scored {away_score} goal(s)."
            f"{stage_text}"
        )

    # -------------------------------------------------
    # MATCH SCORE
    # -------------------------------------------------
    elif intent == "match_score":

        base = (
            f"{home_team} {home_score} - "
            f"{away_score} {away_team}."
        )

        if home_score == away_score:
            shootout = get_shootout_for_match(row)

            if shootout is not None:
                winner = shootout["winner"]

                return (
                    f"{base} "
                    f"{winner} won the penalty shootout."
                    f"{stage_text}"
                )

        return f"{base}{stage_text}"

    # -------------------------------------------------
    # MATCH WINNER
    # -------------------------------------------------
    elif intent == "match_winner":

        if home_score > away_score:
            return (
                f"{home_team} won the match "
                f"{home_score}-{away_score}."
                f"{stage_text}"
            )

        elif away_score > home_score:
            return (
                f"{away_team} won the match "
                f"{away_score}-{home_score}."
                f"{stage_text}"
            )

        shootout = get_shootout_for_match(row)

        if shootout is not None:
            winner = shootout["winner"]

            return (
                f"The match ended "
                f"{home_score}-{away_score}, "
                f"and {winner} won the penalty shootout."
                f"{stage_text}"
            )

        return (
            f"The match ended in a "
            f"{home_score}-{away_score} draw."
            f"{stage_text}"
        )

    # -------------------------------------------------
    # TOTAL GOALS
    # -------------------------------------------------
    elif intent == "total_goals":
        total = home_score + away_score

        return (
            f"A total of {total} goal(s) were scored."
            f"{stage_text}"
        )

    # -------------------------------------------------
    # DATE
    # -------------------------------------------------
    elif intent == "match_date":
        return (
            f"The match was played on {date}."
            f"{stage_text}"
        )

    # -------------------------------------------------
    # LOCATION
    # -------------------------------------------------
    elif intent == "match_location":
        return (
            f"The match was played in "
            f"{city}, {country}."
            f"{stage_text}"
        )

    # -------------------------------------------------
    # TOURNAMENT
    # -------------------------------------------------
    elif intent == "tournament":

        if stage:
            return (
                f"The match was part of the "
                f"{tournament}, in the {stage}."
            )

        return (
            f"The match was part of the "
            f"{tournament}."
        )

    # -------------------------------------------------
    # NEUTRAL STATUS
    # -------------------------------------------------
    elif intent == "neutral_status":
        neutral_text = "Yes" if bool(neutral) else "No"

        return (
            f"{neutral_text}, the match "
            f"{'was' if bool(neutral) else 'was not'} "
            f"played at a neutral venue."
            f"{stage_text}"
        )

    return "Answer generation for this intent is not implemented yet."

In [280]:
football_qa(
    "last match score between france and switzerland"
)


Assistant: I found matches across 38 different years. Do you remember roughly which year it was?

You: 2021

Assistant: France 3 - 3 Switzerland. Switzerland won the penalty shootout. This was a Round of 16 match at the UEFA Euro.


In [281]:
football_qa(
    "who won between argentina and croatia in 2022"
)


Assistant: Argentina won the match 3-0. This was a Semifinal match at the FIFA World Cup.


In [282]:
football_qa(
    "who won between croatia and morocco in 2022"
)


Assistant: Do you remember the city where the match was played? Options: Al Khor, Al Rayyan

You: Al Khor

Assistant: The match ended in a 0-0 draw.


In [283]:
football_qa("Who won the world cup in 2010?")


Assistant: Spain won the FIFA World Cup final 1-0 against Netherlands.


In [284]:
football_qa("What are Argentina's last 5 international matches?")


Assistant: Argentina's last 5 international matches:

1. 2026-07-03: Argentina 3-2 Cape Verde
2. 2026-07-07: Argentina 3-2 Egypt
3. 2026-07-11: Argentina 3-1 Switzerland
4. 2026-07-15: England 1-2 Argentina
5. 2026-07-19: Spain 1-0 Argentina

Form: W-W-W-W-L
Record: 4 wins, 0 draws, 1 losses
Goals: 11 scored, 7 conceded.


In [285]:
football_qa("Who won the penalty shootout between Uruguay and Ghana?")

Possible shootout records: 1

Assistant: Uruguay won the penalty shootout between Uruguay and Ghana on 2010-07-02.


In [286]:
football_qa("last match score between france and switzerland")


Assistant: I found matches across 38 different years. Do you remember roughly which year it was?

You: 2021

Assistant: France 3 - 3 Switzerland. Switzerland won the penalty shootout. This was a Round of 16 match at the UEFA Euro.


In [287]:
football_qa("What was the score between Turkey and Latvia on 2013-05-28?")


Assistant: Turkey 3 - 3 Latvia.


In [288]:
football_qa("Who won the 2014 FIFA World Cup Final?")


Assistant: Germany won the FIFA World Cup final 1-0 against Argentina.


In [289]:
test_questions = [
    "Who won the 2014 FIFA World Cup Final?",
    "What was the score between France and Switzerland at Euro 2020?",
    "Who scored for Paraguay against Chile in 2016?",
    "Who won the penalty shootout between Uruguay and Ghana?"
]

for question in test_questions:
    print("\n" + "=" * 70)
    print("Question:", question)
    football_qa(question)


Question: Who won the 2014 FIFA World Cup Final?

Assistant: Germany won the FIFA World Cup final 1-0 against Argentina.

Question: What was the score between France and Switzerland at Euro 2020?

Assistant: I couldn't find a match that fits the information in your question.

Question: Who scored for Paraguay against Chile in 2016?

Assistant: The scorers were Óscar Romero, Paulo da Silva in the Paraguay vs Chile match on 2016-09-01.

Question: Who won the penalty shootout between Uruguay and Ghana?
Possible shootout records: 1

Assistant: Uruguay won the penalty shootout between Uruguay and Ghana on 2010-07-02.


In [290]:
question = "What was the score between France and Switzerland at Euro 2020?"

print("Predicted intent:", predict_intent(question))

Predicted intent: match_score


In [291]:
print("football_qa exists:", "football_qa" in globals())
print("football_qa_lstm exists:", "football_qa_lstm" in globals())
print("lstm_football_qa exists:", "lstm_football_qa" in globals())

football_qa exists: True
football_qa_lstm exists: False
lstm_football_qa exists: False


In [292]:
football_qa("Who won the 2014 FIFA World Cup final?")


Assistant: Germany won the FIFA World Cup final 1-0 against Argentina.


In [293]:
football_qa("What was the score between Germany and Argentina in the 2014 FIFA World Cup final?")


Assistant: Germany 1 - 0 Argentina. This was a Final match at the FIFA World Cup.


In [294]:
football_qa("Did Georgios Samaras convert a penalty against Germany on 2012-06-22?")


Assistant: Georgios Samaras's goal was not a penalty.


In [295]:
football_qa("Who won the penalty shootout between Uruguay and Ghana?")

Possible shootout records: 1

Assistant: Uruguay won the penalty shootout between Uruguay and Ghana on 2010-07-02.


In [296]:
!pip install -q gradio

import gradio as gr
import io
import re
from contextlib import redirect_stdout


# =========================================================
# 1. LSTM QA -> GRADIO WRAPPER
# =========================================================

def polish_lstm_answer(output):

    cleaned_lines = []

    for line in output.splitlines():
        line = line.strip()

        if not line:
            continue

        # Hide internal debug output
        # Example: Possible shootout records: 1
        if line.startswith("Possible ") and "records:" in line:
            continue

        # Remove notebook console prefix
        if line.startswith("Assistant:"):
            line = line[len("Assistant:"):].strip()

        if line:
            cleaned_lines.append(line)

    answer = "\n".join(cleaned_lines)

    # Small grammar improvements
    answer = re.sub(r"\b1 wins\b", "1 win", answer)
    answer = re.sub(r"\b1 draws\b", "1 draw", answer)
    answer = re.sub(r"\b1 losses\b", "1 loss", answer)

    return answer.strip()


def run_lstm_football_qa(message):

    if not message or not message.strip():
        return "Please enter a football question."

    buffer = io.StringIO()

    try:

        # Use conversational normalization if it exists
        if "normalize_user_question" in globals():
            clean_question = normalize_user_question(message)
        else:
            clean_question = message.strip()

        with redirect_stdout(buffer):
            football_qa(clean_question)

        output = buffer.getvalue().strip()

        if not output:
            return (
                "I couldn't find enough information to answer that question. "
                "Try adding a team, date, tournament, or opponent."
            )

        answer = polish_lstm_answer(output)

        if not answer:
            return (
                "I couldn't find enough information to answer that question."
            )

        return answer

    except Exception:
        return (
            "I couldn't process that question. "
            "Try rephrasing it using a specific match, team, date, or tournament."
        )


# =========================================================
# 2. CHAT
# =========================================================

WELCOME_MESSAGE = {
    "role": "assistant",
    "content": (
        "### Welcome to GLOBAL XI ⚽\n\n"
        "This version of GLOBAL XI is powered by the project's "
        "**LSTM deep learning model**.\n\n"
        "Ask me about **scores, winners, goal events, tournaments, "
        "penalties, shootouts and other historical international "
        "football information**.\n\n"
        "Type a question below or try one of the tested examples."
    )
}


def respond_lstm(message, history):

    if not message or not message.strip():
        return history, ""

    if history is None:
        history = [WELCOME_MESSAGE]

    user_message = message.strip()

    history = history + [
        {
            "role": "user",
            "content": user_message
        }
    ]

    answer = run_lstm_football_qa(user_message)

    history = history + [
        {
            "role": "assistant",
            "content": answer
        }
    ]

    return history, ""


def use_lstm_example(question, history):
    return respond_lstm(question, history)


def reset_lstm_chat():
    return [WELCOME_MESSAGE], ""


# =========================================================
# 3. TESTED LSTM QUESTIONS
# =========================================================

example_1 = (
    "Who won the 2014 FIFA World Cup final?"
)

example_2 = (
    "What was the score between Germany and Argentina "
    "in the 2014 FIFA World Cup final?"
)

example_3 = (
    "Did Georgios Samaras convert a penalty "
    "against Germany on 2012-06-22?"
)

example_4 = (
    "Who won the penalty shootout between Uruguay and Ghana?"
)


# =========================================================
# 4. CSS
# =========================================================

custom_css = """

.gradio-container {
    max-width: 1180px !important;
    margin: 0 auto !important;
    padding: 18px 24px 38px 24px !important;
}

footer {
    display: none !important;
}


/* HERO */

#hero {
    text-align: center;
    padding: 24px 10px 20px 10px;
}

#hero h1 {
    font-size: 2.65rem !important;
    font-weight: 800 !important;
    letter-spacing: -0.04em;
    margin-bottom: 4px !important;
}

#hero h3 {
    font-weight: 450 !important;
    opacity: 0.82;
}


/* CHAT SHELL */

#chat-shell {
    border: 1px solid rgba(120,120,120,0.20);
    border-radius: 26px;
    padding: 14px;

    box-shadow:
        0 16px 45px rgba(0,0,0,0.08);
}


/* CHATBOT */

#chatbot {
    border-radius: 21px !important;
    border: none !important;
    min-height: 470px !important;
}


/* INPUT */

#composer-row {
    margin-top: 12px;
    align-items: center;
    gap: 10px;
}

#question-box textarea {
    border-radius: 18px !important;
    padding: 14px 16px !important;
    font-size: 15px !important;
    min-height: 48px !important;
}


/* SEND */

#send-button {
    border-radius: 17px !important;
    min-width: 105px !important;
    height: 48px !important;
    font-weight: 700 !important;
}


/* RESET */

#reset-button {
    border-radius: 14px !important;
}


/* SUGGESTIONS */

#suggestions-heading {
    margin-top: 28px;
    margin-bottom: 10px;
    text-align: center;
}

.suggestion-btn {
    border-radius: 17px !important;
    text-align: left !important;
    padding: 13px 16px !important;
    min-height: 58px !important;
    font-weight: 520 !important;
    white-space: normal !important;
    transition: 0.15s ease !important;
}

.suggestion-btn:hover {
    transform: translateY(-1px);
}


/* MODEL INFO */

#model-info {
    text-align: center;
    margin-top: 25px;
    padding: 19px;
    border-radius: 19px;
    border: 1px solid rgba(120,120,120,0.15);
}


#footer-note {
    text-align: center;
    opacity: 0.56;
    font-size: 12px;
    margin-top: 15px;
}

"""


# =========================================================
# 5. BUILD INTERFACE
# =========================================================

with gr.Blocks() as lstm_demo:

    # HERO
    gr.Markdown(
        """
# ⚽ GLOBAL XI

### LSTM Football Question Answering Assistant

Ask natural-language questions about historical international football.

**Tokenization • Embedding • LSTM • Factual Football Data Retrieval**
        """,
        elem_id="hero"
    )


    # CHAT
    with gr.Group(elem_id="chat-shell"):

        chatbot = gr.Chatbot(
            value=[WELCOME_MESSAGE],
            height=470,
            elem_id="chatbot"
        )

        with gr.Row(elem_id="composer-row"):

            question_box = gr.Textbox(
                placeholder="Ask GLOBAL XI a football question...",
                show_label=False,
                lines=1,
                autofocus=True,
                scale=9,
                elem_id="question-box"
            )

            send_button = gr.Button(
                "Send ➜",
                variant="primary",
                scale=1,
                elem_id="send-button"
            )

        reset_button = gr.Button(
            "↻  New conversation",
            variant="secondary",
            elem_id="reset-button"
        )


    # =====================================================
    # SUGGESTIONS
    # =====================================================

    gr.Markdown(
        """
### ✨ Try a question

Choose one of these tested LSTM examples:
        """,
        elem_id="suggestions-heading"
    )


    with gr.Row():

        example_button_1 = gr.Button(
            "🏆 Who won the 2014 FIFA World Cup final?",
            elem_classes=["suggestion-btn"]
        )

        example_button_2 = gr.Button(
            "⚽ What was the score between Germany and Argentina "
            "in the 2014 FIFA World Cup final?",
            elem_classes=["suggestion-btn"]
        )


    with gr.Row():

        example_button_3 = gr.Button(
            "🎯 Did Georgios Samaras convert a penalty "
            "against Germany on 2012-06-22?",
            elem_classes=["suggestion-btn"]
        )

        example_button_4 = gr.Button(
            "🥅 Who won the penalty shootout between Uruguay and Ghana?",
            elem_classes=["suggestion-btn"]
        )


    # =====================================================
    # MODEL INFORMATION
    # =====================================================

    gr.Markdown(
        """
### Deep Learning Pipeline

**User Question** → **Tokenizer** → **Padded Sequence** →
**Embedding Layer** → **LSTM Intent Classification** →
**Football Data Retrieval** → **Final Answer**

This interface demonstrates the project's trained **LSTM model**
integrated with the football question-answering pipeline.
        """,
        elem_id="model-info"
    )


    gr.Markdown(
        """
GLOBAL XI uses the trained LSTM intent classifier together with
processed historical international football datasets to retrieve
factual answers.
        """,
        elem_id="footer-note"
    )


    # =====================================================
    # 6. EVENTS
    # =====================================================

    # Send button
    send_button.click(
        fn=respond_lstm,
        inputs=[question_box, chatbot],
        outputs=[chatbot, question_box]
    )

    # Enter key
    question_box.submit(
        fn=respond_lstm,
        inputs=[question_box, chatbot],
        outputs=[chatbot, question_box]
    )

    # Reset
    reset_button.click(
        fn=reset_lstm_chat,
        inputs=None,
        outputs=[chatbot, question_box]
    )

    # Example 1
    example_button_1.click(
        fn=lambda history: use_lstm_example(example_1, history),
        inputs=[chatbot],
        outputs=[chatbot, question_box]
    )

    # Example 2
    example_button_2.click(
        fn=lambda history: use_lstm_example(example_2, history),
        inputs=[chatbot],
        outputs=[chatbot, question_box]
    )

    # Example 3
    example_button_3.click(
        fn=lambda history: use_lstm_example(example_3, history),
        inputs=[chatbot],
        outputs=[chatbot, question_box]
    )

    # Example 4
    example_button_4.click(
        fn=lambda history: use_lstm_example(example_4, history),
        inputs=[chatbot],
        outputs=[chatbot, question_box]
    )


# =========================================================
# 7. LAUNCH
# =========================================================

lstm_demo.queue()

lstm_demo.launch(
    share=True,
    theme=gr.themes.Soft(),
    css=custom_css
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f67b0195c32c7aab5f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
